# Free-Form Rationale Evidence Consolidation

This notebook evaluates the **Free-form rationale** consolidation format used in the thesis output-format comparison.

It follows the Binary-only and Structured R1 consolidation experiments while keeping the same underlying 400-case development set and the same participant-centric evidence packet.

## Relationship to Structured R1

Structured R1 requires the model to expose a fixed set of categorical diagnostic fields before returning the final binary label:

- participation assessment,
- local temporal assessment,
- global temporal assessment,
- combined temporal assessment,
- semantic assessment,
- decisive evidence dimension,
- final `NORMAL` / `ANOMALOUS` prediction.

The **Free-form rationale** experiment removes those structured categorical fields.

Instead, the model returns:

```text
free-form rationale
        ↓
binary label
```

However, the rationale is **not unconstrained**.

The prompt explicitly instructs the model to reason through the same evidence dimensions that motivated Structured R1:

1. assess participation;
2. assess local temporal evidence;
3. assess global temporal evidence;
4. combine local and global temporal evidence using the existing temporal policy;
5. assess semantic compatibility;
6. identify the decisive evidence dimension;
7. explain how the final binary label follows from the existing decision policy.

The key experimental difference is therefore the **output scaffold**.

Structured R1 forces each evidence dimension into a fixed categorical schema, whereas this experiment allows those same assessments to be expressed in a concise natural-language rationale before the final label.

No structured assessment fields are requested or returned.

## Controlled comparison

The experiment preserves the model-facing components of the Binary-only consolidation source setup:

- the same 400 cases;
- the same Qwen2.5-Omni-7B checkpoint;
- the same deterministic decoding policy;
- the same frozen NORMAL temporal reference;
- the same participation evidence;
- the same filtered turns;
- the same local temporal features;
- the same global temporal features;
- the same coarse semantic summaries;
- the same focused semantic summaries;
- the same independent-normality decision policy.

The original binary output block is replaced by a rationale-first output block. The model is asked to produce one concise free-form rationale followed by the binary label.

The experiment also verifies that:

- a rationale is produced for every case;
- the rationale appears before the final label;
- no Structured R1 categorical fields are emitted.

## Thesis-reported result

The Free-form rationale configuration produces:

| Format | NORMAL | LAG | WRONG | SILENT | Accuracy |
|---|---:|---:|---:|---:|---:|
| **Free-form rationale** | **90/100** | **43/100** | **46/100** | **100/100** | **69.75%** |

The resulting confusion matrix is:

```text
                Pred NORMAL   Pred ANOMALOUS
Gold NORMAL          90             10
Gold ANOMALOUS      111            189
```

Compared with Structured R1, the free-form rationale strongly favours broad conversational compatibility:

- NORMAL preservation increases,
- LAG sensitivity decreases substantially,
- Wrong Partner sensitivity decreases substantially,
- Silent Partner remains perfectly detected.

This result is important for the thesis because the **underlying evidence remains fixed while the output scaffold changes**, yet classification behaviour changes markedly. The consolidation format is therefore part of the effective decision mechanism rather than merely a passive reporting choice.

The rationale outputs and the subsequent inspection cells are retained as diagnostic reports of observable model behaviour. They should not be interpreted as guaranteed faithful descriptions of the model's hidden internal reasoning.

> **Reproducibility note:** all code cells, execution counts, code-cell metadata, cached outputs, evaluation results, and inspection outputs are preserved exactly as executed. Only this introductory Markdown cell has been rewritten for the public repository.

## 1. Install dependencies

Run this cell once in a fresh Colab runtime. Then select:

`Runtime → Restart session`

and continue from the next cell.

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece
!pip install -U qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 147.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 21.7 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Mount Google Drive and configure artifact paths

The folder name still contains `1_2_3sec` for historical reasons. The finalized 400-case database is loaded from the exact path below.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
from collections import Counter
import copy
import json

import pandas as pd
from IPython.display import display


# ============================================================
# MAIN PROJECT DIRECTORY
# ============================================================

OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)


# ============================================================
# INPUT ARTIFACTS
# ============================================================

REFERENCE_BASE_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_base_statistics.json"
)


REFERENCE_SHIFT_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_global_shift_statistics.json"
)


FINAL_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)


# ============================================================
# MODEL
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"


MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)


# ============================================================
# NEW EXPERIMENT DIRECTORY
#
# This is intentionally different from the previous
# chain-of-thought-before-label experiment, so no prior result
# or cache can be overwritten.
# ============================================================

LABEL_BEFORE_RATIONALE_EXPERIMENT_DIR = (
    OUT_DIR
    / (
        "reasoning_r1_full_semantics_"
        "normal_references_only_"
        "label_before_chain_of_thought"
    )
)


print("=" * 88)
print("ARTIFACT PATH AUDIT")
print("=" * 88)


required_paths = {
    "Frozen base statistics": (
        REFERENCE_BASE_STATS_PATH
    ),

    "Frozen global-shift statistics": (
        REFERENCE_SHIFT_STATS_PATH
    ),

    "Final 400-case database": (
        FINAL_DATABASE_PATH
    ),

    "Local Qwen checkpoint": (
        MODEL_PATH
    ),
}


for name, path in required_paths.items():

    print(
        f"{name}:",
        path,
    )

    print(
        "  exists:",
        path.exists(),
    )


print(
    "New experiment directory:",
    LABEL_BEFORE_RATIONALE_EXPERIMENT_DIR,
)


print(
    "Existing resumable cache:",
    (
        LABEL_BEFORE_RATIONALE_EXPERIMENT_DIR
        / "predictions_cache.json"
    ).exists(),
)


# ============================================================
# SAFETY CHECKS
# ============================================================

assert OUT_DIR.exists(), (
    f"Project directory not found: {OUT_DIR}"
)


assert REFERENCE_BASE_STATS_PATH.exists(), (
    "Frozen base-statistics file not found: "
    f"{REFERENCE_BASE_STATS_PATH}"
)


assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    "Frozen global-shift-statistics file not found: "
    f"{REFERENCE_SHIFT_STATS_PATH}"
)


assert FINAL_DATABASE_PATH.exists(), (
    "Final consolidation database not found: "
    f"{FINAL_DATABASE_PATH}"
)


assert MODEL_PATH.exists(), (
    "Local Qwen checkpoint not found: "
    f"{MODEL_PATH}"
)


Mounted at /content/drive
ARTIFACT PATH AUDIT
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True
New experiment directory: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_label_before_chain_of_thought
Existing resumable cache: True


## 3. Load the frozen NORMAL reference statistics

Both frozen JSON files contain separate reference profiles. This experiment deliberately extracts and retains only:

```text
reference_profile = NORMAL
```

The lag reference profiles are not included in the binary-only experiment state at this stage.

In [ ]:
# ============================================================
# LOAD FROZEN STATISTICS AND RETAIN NORMAL ONLY
# ============================================================

def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected a JSON list in {path}"
    )


    return data


frozen_base_records_all_profiles = (
    load_json_list(
        REFERENCE_BASE_STATS_PATH
    )
)


frozen_shift_records_all_profiles = (
    load_json_list(
        REFERENCE_SHIFT_STATS_PATH
    )
)


base_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_base_records_all_profiles
]


shift_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_shift_records_all_profiles
]


assert len(
    base_profiles
) == len(
    set(
        base_profiles
    )
)


assert len(
    shift_profiles
) == len(
    set(
        shift_profiles
    )
)


assert "NORMAL" in base_profiles
assert "NORMAL" in shift_profiles


normal_base_matches = [
    record

    for record
    in frozen_base_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


normal_shift_matches = [
    record

    for record
    in frozen_shift_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


assert len(
    normal_base_matches
) == 1


assert len(
    normal_shift_matches
) == 1


frozen_normal_base_statistics = copy.deepcopy(
    normal_base_matches[0]
)


frozen_normal_shift_statistics = copy.deepcopy(
    normal_shift_matches[0]
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# This is the only frozen reference object that should later
# be passed to the binary-only prompt constructor.
FROZEN_NORMAL_REFERENCE = {
    "base_statistics": (
        frozen_normal_base_statistics
    ),

    "global_shift_statistics": (
        frozen_normal_shift_statistics
    ),
}


normal_base_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_base_statistics.items()

    if key != "reference_profile"
])


normal_shift_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_shift_statistics.items()

    if key != "reference_profile"
])


print("=" * 88)
print("FROZEN REFERENCE FILES LOADED")
print("=" * 88)

print(
    "Profiles stored in base-statistics file:",
    base_profiles,
)

print(
    "Profiles stored in global-shift file:",
    shift_profiles,
)


print("\n" + "=" * 88)
print("FROZEN NORMAL BASE TEMPORAL STATISTICS")
print("=" * 88)

display(
    normal_base_display_df
)


print("\n" + "=" * 88)
print("FROZEN NORMAL GLOBAL-SHIFT STATISTICS")
print("=" * 88)

display(
    normal_shift_display_df
)


print(
    "\nOnly NORMAL frozen references retained:",
    list(
        FROZEN_NORMAL_REFERENCE.keys()
    ),
)

FROZEN REFERENCE FILES LOADED
Profiles stored in base-statistics file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']
Profiles stored in global-shift file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']

FROZEN NORMAL BASE TEMPORAL STATISTICS


,metric,value
0,num_original_A_turns,11.800000
1,num_original_B_turns,11.080000
2,num_removed_A_backchannels,1.740000
3,num_removed_B_backchannels,1.660000
4,num_filtered_A_turns,10.060000
5,num_filtered_B_turns,9.420000
6,num_offsets,5.760000
7,num_negative,2.040000
8,num_positive,3.720000
9,offset_mean,0.302128



FROZEN NORMAL GLOBAL-SHIFT STATISTICS


,metric,value
0,best_B_correction_shift_seconds__mean,-0.230000
1,best_B_correction_shift_seconds__median,-0.000000
2,estimated_B_lateness_seconds__mean,0.500000
3,estimated_B_lateness_seconds__median,0.000000
4,alignment_score_gain_vs_zero__mean,0.023376
5,alignment_score_gain_vs_zero__median,0.009105
6,best_num_bilateral_events__mean,11.940000
7,best_event_coverage__mean,0.595175



Only NORMAL frozen references retained: ['base_statistics', 'global_shift_statistics']


## 4. Load and audit the final 400-case database

This cell verifies:

- exactly 400 unique cases;
- exactly 100 cases per family;
- 100 `NORMAL` and 300 `ANOMALOUS` binary labels;
- participant-level `speaks` agrees with the final filtered VAD turns;
- every sample has all four coarse and all four focused summaries;
- no focused summary contains the legacy `speaks` field.

In [ ]:
# ============================================================
# LOAD FINAL DATABASE
# ============================================================

consolidation_cases = load_json_list(
    FINAL_DATABASE_PATH
)


assert len(
    consolidation_cases
) == 400


# ============================================================
# NORMALIZE CASE FAMILY
# ============================================================

def get_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case_variant: "
        f"{case.get('case_variant')}"
    )


# ============================================================
# GLOBAL COUNTS
# ============================================================

case_ids = [
    str(
        case[
            "case_id"
        ]
    )

    for case
    in consolidation_cases
]


assert len(
    case_ids
) == len(
    set(
        case_ids
    )
)


family_counts = Counter(
    get_case_family(
        case
    )

    for case
    in consolidation_cases
)


expected_family_counts = {
    "normal": 100,
    "wrong_partner": 100,
    "lag": 100,
    "silent_partner": 100,
}


assert dict(
    family_counts
) == expected_family_counts


binary_label_counts = Counter(
    str(
        case[
            "gold_binary_label"
        ]
    )

    for case
    in consolidation_cases
)


assert binary_label_counts[
    "NORMAL"
] == 100


assert binary_label_counts[
    "ANOMALOUS"
] == 300


lag_variant_counts = Counter(
    str(
        case.get(
            "case_variant"
        )
    )

    for case
    in consolidation_cases

    if get_case_family(
        case
    ) == "lag"
)


# ============================================================
# PARTICIPANT AND SEMANTIC AUDITS
# ============================================================

SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


num_participant_records_checked = 0
num_semantic_slots_checked = 0


for case in consolidation_cases:

    for (
        role,
        turns_key,
    ) in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),

        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:

        participant = case[
            role
        ]


        turns = case[
            turns_key
        ]


        assert isinstance(
            participant,
            dict,
        )


        assert isinstance(
            turns,
            list,
        )


        assert (
            "speaks"
            in participant
        )


        assert isinstance(
            participant[
                "speaks"
            ],
            bool,
        )


        assert (
            participant[
                "speaks"
            ]
            ==
            (
                len(
                    turns
                ) > 0
            )
        ), (
            "Participant-level speaks disagrees with "
            f'VAD turns in {case["case_id"]} / {role}'
        )


        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        for segment_name in SEGMENT_NAMES:

            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )


            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            assert isinstance(
                coarse_summary,
                dict,
            ), (
                "Missing coarse summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert isinstance(
                focused_summary,
                dict,
            ), (
                "Missing focused summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert (
                "speaks"
                not in focused_summary
            ), (
                "Legacy focused-summary speaks field found in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            num_semantic_slots_checked += 1


        num_participant_records_checked += 1


# ============================================================
# DISPLAY DATABASE COMPOSITION
# ============================================================

family_count_df = pd.DataFrame([
    {
        "case_family": family,
        "num_cases": count,
    }

    for family, count
    in sorted(
        family_counts.items()
    )
])


binary_count_df = pd.DataFrame([
    {
        "gold_binary_label": label,
        "num_cases": count,
    }

    for label, count
    in sorted(
        binary_label_counts.items()
    )
])


lag_variant_df = pd.DataFrame([
    {
        "case_variant": variant,
        "num_cases": count,
    }

    for variant, count
    in sorted(
        lag_variant_counts.items()
    )
])


print("=" * 88)
print("FINAL CONSOLIDATION DATABASE AUDIT PASSED")
print("=" * 88)

print(
    "Database:",
    FINAL_DATABASE_PATH,
)

print(
    "Total unique cases:",
    len(
        consolidation_cases
    ),
)

print(
    "Participant records checked:",
    num_participant_records_checked,
)

print(
    "Semantic participant-segment slots checked:",
    num_semantic_slots_checked,
)


print("\nCASE FAMILIES")

display(
    family_count_df
)


print("\nBINARY LABELS")

display(
    binary_count_df
)


print("\nLAG VARIANTS STORED IN THE FINAL DATABASE")

display(
    lag_variant_df
)

FINAL CONSOLIDATION DATABASE AUDIT PASSED
Database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Total unique cases: 400
Participant records checked: 800
Semantic participant-segment slots checked: 1600

CASE FAMILIES


,case_family,num_cases
0,lag,100
1,normal,100
2,silent_partner,100
3,wrong_partner,100



BINARY LABELS


,gold_binary_label,num_cases
0,ANOMALOUS,300
1,NORMAL,100



LAG VARIANTS STORED IN THE FINAL DATABASE


,case_variant,num_cases
0,lag_2sec,50
1,lag_3sec,50


## 5. Participant-level `speaks` overview

The field is derived exclusively from each sample's final filtered VAD turns.

In [ ]:
# ============================================================
# PARTICIPANT-LEVEL SPEAKS SUMMARY
# ============================================================

speaks_rows = []


for case in consolidation_cases:

    speaks_rows.append({
        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_id": (
            case[
                "case_id"
            ]
        ),

        "participant_A_speaks": (
            case[
                "participant_A"
            ][
                "speaks"
            ]
        ),

        "participant_B_speaks": (
            case[
                "participant_B"
            ][
                "speaks"
            ]
        ),

        "participant_A_num_turns": len(
            case[
                "participant_A_filtered_turns"
            ]
        ),

        "participant_B_num_turns": len(
            case[
                "participant_B_filtered_turns"
            ]
        ),
    })


speaks_df = pd.DataFrame(
    speaks_rows
)


speaks_summary_df = (
    speaks_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        participant_A_speaks_true=(
            "participant_A_speaks",
            "sum",
        ),

        participant_B_speaks_true=(
            "participant_B_speaks",
            "sum",
        ),
    )
)


speaks_summary_df[
    "participant_A_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_A_speaks_true"
    ]
)


speaks_summary_df[
    "participant_B_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_B_speaks_true"
    ]
)


speaks_summary_df = speaks_summary_df[
    [
        "case_family",
        "total_cases",

        "participant_A_speaks_true",
        "participant_A_speaks_false",

        "participant_B_speaks_true",
        "participant_B_speaks_false",
    ]
]


display(
    speaks_summary_df
)

,case_family,total_cases,participant_A_speaks_true,participant_A_speaks_false,participant_B_speaks_true,participant_B_speaks_false
0,lag,100,100,0,100,0
1,normal,100,100,0,100,0
2,silent_partner,100,100,0,0,100
3,wrong_partner,100,100,0,100,0


## 6. Interactive case inspection

The inspector supports all four families:

- `NORMAL`
- `WRONG_PARTNER`
- `LAG`
- `SILENT_PARTNER`

It can display participant metadata, VAD-derived `speaks`, filtered turns, temporal features, semantic summaries, and the complete sample JSON.

The direct function can also be used without widgets:

```python
inspect_case(
    family="normal",
    sample_index=0,
)
```

In [ ]:
# ============================================================
# INTERACTIVE DATABASE INSPECTOR
# ============================================================

import ipywidgets as widgets

from IPython.display import (
    clear_output,
    display,
)


try:

    from google.colab import output

    output.enable_custom_widget_manager()

except Exception:

    pass


# ============================================================
# SPLIT AND SORT CASES
# ============================================================

cases_by_family = {
    "normal": [],
    "wrong_partner": [],
    "lag": [],
    "silent_partner": [],
}


for case in consolidation_cases:

    cases_by_family[
        get_case_family(
            case
        )
    ].append(
        case
    )


for family in cases_by_family:

    cases_by_family[
        family
    ] = sorted(
        cases_by_family[
            family
        ],

        key=lambda case: str(
            case.get(
                "case_id",
                "",
            )
        ),
    )


# ============================================================
# DISPLAY HELPERS
# ============================================================

def participant_row(
    case,
    role,
):
    participant = case[
        role
    ]


    turns_key = (
        f"{role}_filtered_turns"
    )


    turns = case.get(
        turns_key,
        [],
    )


    return {
        "role": role,

        "conversation_id": (
            participant.get(
                "conversation_id"
            )
        ),

        "participant_id": (
            participant.get(
                "participant_id"
            )
        ),

        "speaks": (
            participant.get(
                "speaks"
            )
        ),

        "num_filtered_turns": len(
            turns
        ),

        "metadata_path": (
            participant.get(
                "metadata_path"
            )
        ),
    }


def display_semantics(
    case,
    role,
):
    print(
        "\n" + "-" * 88
    )

    print(
        f"{role.upper()} SEMANTIC SUMMARIES"
    )

    print(
        "-" * 88
    )


    participant_semantics = (
        case[
            "semantic_summaries"
        ][role]
    )


    for segment_name in SEGMENT_NAMES:

        segment_record = (
            participant_semantics[
                segment_name
            ]
        )


        print(
            f"\nSEGMENT: {segment_name}"
        )


        print(
            "\nCoarse summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "coarse_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\nFocused summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "focused_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


# ============================================================
# MAIN INSPECTION FUNCTION
# ============================================================

def inspect_case(
    family,
    sample_index,
    show_turns=True,
    show_temporal_features=True,
    show_semantics=True,
    show_full_json=False,
):
    assert family in cases_by_family


    family_cases = cases_by_family[
        family
    ]


    sample_index = int(
        sample_index
    )


    assert (
        0
        <= sample_index
        < len(
            family_cases
        )
    )


    case = family_cases[
        sample_index
    ]


    print("=" * 88)
    print("CONSOLIDATION CASE INSPECTION")
    print("=" * 88)

    print(
        "Family:",
        family,
    )

    print(
        "Family position:",
        f"{sample_index + 1}/{len(family_cases)}",
    )

    print(
        "case_id:",
        case.get(
            "case_id"
        ),
    )

    print(
        "source_group_id:",
        case.get(
            "source_group_id"
        ),
    )

    print(
        "case_variant:",
        case.get(
            "case_variant"
        ),
    )

    print(
        "gold_binary_label:",
        case.get(
            "gold_binary_label"
        ),
    )

    print(
        "gold_anomaly_type:",
        case.get(
            "gold_anomaly_type"
        ),
    )

    print(
        "A_source_conversation:",
        case.get(
            "A_source_conversation"
        ),
    )

    print(
        "B_source_conversation:",
        case.get(
            "B_source_conversation"
        ),
    )


    print(
        "\nPARTICIPANTS"
    )


    display(
        pd.DataFrame([
            participant_row(
                case,
                "participant_A",
            ),

            participant_row(
                case,
                "participant_B",
            ),
        ])
    )


    print(
        "\nSEMANTIC COVERAGE"
    )


    print(
        json.dumps(
            case.get(
                "semantic_summary_coverage"
            ),
            indent=2,
            ensure_ascii=False,
        )
    )


    if show_turns:

        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT A FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_A_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT B FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_B_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_temporal_features:

        print(
            "\n" + "=" * 88
        )

        print(
            "LOCAL TEMPORAL FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "local_temporal_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "GLOBAL SHIFT FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "global_shift_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_semantics:

        display_semantics(
            case,
            "participant_A",
        )


        display_semantics(
            case,
            "participant_B",
        )


    if show_full_json:

        print(
            "\n" + "=" * 88
        )

        print(
            "COMPLETE SAMPLE JSON"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case,
                indent=2,
                ensure_ascii=False,
            )
        )


    return case


# ============================================================
# WIDGETS
# ============================================================

family_labels = {
    "normal": "NORMAL",
    "wrong_partner": "WRONG_PARTNER",
    "lag": "LAG",
    "silent_partner": "SILENT_PARTNER",
}


family_dropdown = widgets.Dropdown(
    options=[
        (
            family_labels[
                family
            ],
            family,
        )

        for family in [
            "normal",
            "wrong_partner",
            "lag",
            "silent_partner",
        ]
    ],

    value="normal",

    description="Family:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="450px"
    ),
)


case_dropdown = widgets.Dropdown(
    description="Sample:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="900px"
    ),
)


show_turns_checkbox = widgets.Checkbox(
    value=True,
    description="Show filtered turns",
    indent=False,
)


show_temporal_checkbox = widgets.Checkbox(
    value=True,
    description="Show temporal features",
    indent=False,
)


show_semantics_checkbox = widgets.Checkbox(
    value=True,
    description="Show semantic summaries",
    indent=False,
)


show_full_json_checkbox = widgets.Checkbox(
    value=False,
    description="Show complete JSON",
    indent=False,
)


inspector_output = widgets.Output()


def refresh_case_options(
    *args,
):
    family = family_dropdown.value


    family_cases = cases_by_family[
        family
    ]


    case_dropdown.options = [
        (
            (
                f"{index + 1:03d}/100 | "
                f'{case.get("case_id")} | '
                f'{case.get("case_variant")}'
            ),

            index,
        )

        for index, case
        in enumerate(
            family_cases
        )
    ]


    case_dropdown.value = 0


def render_case(
    *args,
):
    if case_dropdown.value is None:
        return


    with inspector_output:

        clear_output(
            wait=True
        )


        inspect_case(
            family=(
                family_dropdown.value
            ),

            sample_index=(
                case_dropdown.value
            ),

            show_turns=(
                show_turns_checkbox.value
            ),

            show_temporal_features=(
                show_temporal_checkbox.value
            ),

            show_semantics=(
                show_semantics_checkbox.value
            ),

            show_full_json=(
                show_full_json_checkbox.value
            ),
        )


family_dropdown.observe(
    refresh_case_options,
    names="value",
)


family_dropdown.observe(
    render_case,
    names="value",
)


case_dropdown.observe(
    render_case,
    names="value",
)


show_turns_checkbox.observe(
    render_case,
    names="value",
)


show_temporal_checkbox.observe(
    render_case,
    names="value",
)


show_semantics_checkbox.observe(
    render_case,
    names="value",
)


show_full_json_checkbox.observe(
    render_case,
    names="value",
)


refresh_case_options()


controls = widgets.VBox([
    family_dropdown,
    case_dropdown,

    widgets.HBox([
        show_turns_checkbox,
        show_temporal_checkbox,
    ]),

    widgets.HBox([
        show_semantics_checkbox,
        show_full_json_checkbox,
    ]),
])


print("=" * 88)
print("INTERACTIVE INSPECTOR READY")
print("=" * 88)

for family in [
    "normal",
    "wrong_partner",
    "lag",
    "silent_partner",
]:

    print(
        family,
        len(
            cases_by_family[
                family
            ]
        ),
    )


display(
    controls,
    inspector_output,
)


render_case()


INTERACTIVE INSPECTOR READY
normal 100
wrong_partner 100
lag 100
silent_partner 100


Output()

## 7. Load Qwen2.5-Omni Thinker


In [ ]:
import torch
from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

print("Loaded:", MODEL_ID)

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/233k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Omni-7B


# Ηelper Functions For Qwen Calls

In [ ]:
def extract_json_from_text(text: str):
    text = text.strip()

    # remove markdown fences
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"```$", "", text.strip())

    try:
        return json.loads(text)
    except Exception:
        pass

    # find first balanced JSON object
    start = text.find("{")
    if start == -1:
        return {"parse_error": True, "raw_output": text}

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {"parse_error": True, "raw_output": text}

def qwen_video_text(video_path: str, prompt: str, max_new_tokens: int = 350):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are Qwen, a virtual human developed by the Qwen Team, "
                        "Alibaba Group, capable of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    )
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "video": video_path, "fps": 7.0},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    )

    inputs = inputs.to(model.device)

    print("Input tokens:", inputs["input_ids"].shape[1])

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated

def qwen_text_only(prompt: str, max_new_tokens: int = 700):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={
            "padding": True,
        },
    )

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    input_token_count = int(
        inputs["input_ids"].shape[-1]
    )


    print(
        "Input tokens:",
        input_token_count,
    )


    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


# Frozen NORMAL reference text

This is the same reference-text construction used by the original binary Experiment 2 and the original R1 experiment. No anomaly-specific reference profile is added.


In [ ]:
# ============================================================
# BUILD AND PRINT THE SELECTED NORMAL-ONLY REFERENCE TEXT
#
# Only the frozen NORMAL profile is used.
#
# Included base metrics:
#   - num_offsets
#   - offset_mean
#   - offset_median
#   - offset_max
#   - offset_p75
#   - offset_p90
#   - percent_above_1_5
#   - clean_overlap_seconds
#
# Included global metrics:
#   - best_B_correction_shift_seconds__mean
#   - best_B_correction_shift_seconds__median
#   - estimated_B_lateness_seconds__mean
#   - estimated_B_lateness_seconds__median
#   - alignment_score_gain_vs_zero__mean
#   - alignment_score_gain_vs_zero__median
#   - best_num_bilateral_events__mean
#   - best_event_coverage__mean
#
# No other frozen statistics are exposed.
# ============================================================

import json
import math
import numbers


# ============================================================
# STRICT NORMAL PROFILE AUDIT
# ============================================================

assert isinstance(
    frozen_normal_base_statistics,
    dict,
)


assert isinstance(
    frozen_normal_shift_statistics,
    dict,
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# ============================================================
# NUMERIC VALIDATION HELPER
# ============================================================

def require_finite_numeric(
    record,
    field_name,
):
    """
    Read one required numeric field and verify
    that it exists and contains a finite number.
    """

    assert field_name in record, (
        f"Missing required field: {field_name}"
    )


    value = record[
        field_name
    ]


    assert isinstance(
        value,
        numbers.Real,
    ) and not isinstance(
        value,
        bool,
    ), (
        f"Expected numeric value for {field_name}, "
        f"found {type(value).__name__}: {value}"
    )


    value = float(
        value
    )


    assert math.isfinite(
        value
    ), (
        f"Non-finite value for {field_name}: {value}"
    )


    return value


def normalize_negative_zero(
    value,
):
    """
    Convert -0.0 to 0.0 for cleaner prompt text.
    """

    if abs(
        value
    ) < 1e-12:

        return 0.0


    return value


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL BASE STATISTICS
# ============================================================

normal_num_offsets = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "num_offsets",
    )
)


normal_offset_mean = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_mean",
    )
)


normal_offset_median = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_median",
    )
)


normal_offset_max = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_max",
    )
)


normal_offset_p75 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p75",
    )
)


normal_offset_p90 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p90",
    )
)


normal_percent_above_1_5 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "percent_above_1_5",
    )
)


normal_clean_overlap_seconds = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "clean_overlap_seconds",
    )
)


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL GLOBAL-SHIFT STATISTICS
#
# IMPORTANT:
# These fields use a double underscore before
# "mean" and "median".
# ============================================================

normal_best_shift_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__mean",
    )
)


normal_best_shift_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__median",
    )
)


normal_lateness_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__mean",
    )
)


normal_lateness_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__median",
    )
)


normal_alignment_gain_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__mean",
    )
)


normal_alignment_gain_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__median",
    )
)


normal_bilateral_events_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_num_bilateral_events__mean",
    )
)


normal_event_coverage_fraction = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_event_coverage__mean",
    )
)


# ============================================================
# CLEAN DISPLAY VALUES
# ============================================================

normal_best_shift_mean = (
    normalize_negative_zero(
        normal_best_shift_mean
    )
)


normal_best_shift_median = (
    normalize_negative_zero(
        normal_best_shift_median
    )
)


normal_lateness_mean = (
    normalize_negative_zero(
        normal_lateness_mean
    )
)


normal_lateness_median = (
    normalize_negative_zero(
        normal_lateness_median
    )
)


normal_event_coverage_percent = (
    100.0
    * normal_event_coverage_fraction
)


# ============================================================
# STORE THE EXACT VALUES USED IN THE PROMPT
# ============================================================

normal_base_reference_values = {
    "num_signed_offsets": (
        normal_num_offsets
    ),

    "mean_signed_offset_seconds": (
        normal_offset_mean
    ),

    "median_signed_offset_seconds": (
        normal_offset_median
    ),

    "maximum_signed_offset_seconds": (
        normal_offset_max
    ),

    "p75_signed_offset_seconds": (
        normal_offset_p75
    ),

    "p90_signed_offset_seconds": (
        normal_offset_p90
    ),

    "percent_offsets_above_1_5_seconds": (
        normal_percent_above_1_5
    ),

    "filtered_clean_overlap_seconds": (
        normal_clean_overlap_seconds
    ),
}


normal_global_reference_values = {
    "mean_best_B_correction_shift_seconds": (
        normal_best_shift_mean
    ),

    "median_best_B_correction_shift_seconds": (
        normal_best_shift_median
    ),

    "mean_estimated_B_lateness_seconds": (
        normal_lateness_mean
    ),

    "median_estimated_B_lateness_seconds": (
        normal_lateness_median
    ),

    "mean_alignment_score_gain_vs_zero": (
        normal_alignment_gain_mean
    ),

    "median_alignment_score_gain_vs_zero": (
        normal_alignment_gain_median
    ),

    "mean_num_bilateral_alignment_events": (
        normal_bilateral_events_mean
    ),

    "mean_bilateral_event_coverage_percent": (
        normal_event_coverage_percent
    ),
}


assert len(
    normal_base_reference_values
) == 8


assert len(
    normal_global_reference_values
) == 8


# ============================================================
# BUILD NORMAL LOCAL-TEMPORAL REFERENCE TEXT
# ============================================================

normal_base_reference_text = f"""
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around {normal_num_offsets:.2f}.
- Mean signed offset is around {normal_offset_mean:.2f} seconds.
- Median signed offset is around {normal_offset_median:.2f} seconds.
- Maximum signed offset is around {normal_offset_max:.2f} seconds.
- P75 signed offset is around {normal_offset_p75:.2f} seconds.
- P90 signed offset is around {normal_offset_p90:.2f} seconds.
- Approximately {normal_percent_above_1_5:.1f}% of offsets are above 1.5 seconds.
- Filtered clean overlap is around {normal_clean_overlap_seconds:.2f} seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.
""".strip()


# ============================================================
# BUILD NORMAL GLOBAL-ALIGNMENT REFERENCE TEXT
# ============================================================

normal_global_reference_text = f"""
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around {normal_best_shift_mean:.2f} seconds.
- Median best B correction shift is around {normal_best_shift_median:.2f} seconds.
- Mean estimated B lateness is around {normal_lateness_mean:.2f} seconds.
- Median estimated B lateness is around {normal_lateness_median:.2f} seconds.
- Mean alignment score gain versus zero shift is around {normal_alignment_gain_mean:.3f}.
- Median alignment score gain versus zero shift is around {normal_alignment_gain_median:.3f}.
- Mean number of bilateral alignment events is around {normal_bilateral_events_mean:.2f}.
- Mean bilateral event coverage is around {normal_event_coverage_percent:.2f}%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.
""".strip()


# ============================================================
# PRINT BOTH PROMPT SECTIONS
# ============================================================

print("=" * 88)
print("normal_base_reference_text")
print("=" * 88)

print(
    normal_base_reference_text
)


print("\n" + "=" * 88)
print("normal_global_reference_text")
print("=" * 88)

print(
    normal_global_reference_text
)


# ============================================================
# PRINT THE EXACT STRUCTURED VALUES USED
# ============================================================

print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL BASE VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_base_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL GLOBAL VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_global_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# STRICT CONTENT AUDIT
# ============================================================

combined_reference_text = (
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# No anomaly-specific labels or profiles.
for forbidden_text in [
    "LAG +1",
    "LAG +2",
    "LAG +3",
    "wrong_partner",
    "silent_partner",
    "anomaly_type",
]:

    assert (
        forbidden_text.lower()
        not in combined_reference_text.lower()
    )


# Unwanted frozen reference fields must not appear.
for excluded_text in [
    "original A turns",
    "original B turns",
    "removed A backchannels",
    "removed B backchannels",
    "filtered A turns",
    "filtered B turns",
    "number of negative",
    "number of positive",
    "minimum signed offset",
    "clean overlap percentage",
]:

    assert (
        excluded_text.lower()
        not in combined_reference_text.lower()
    )


# All required selected metrics must appear.
for required_text in [
    "Number of signed offsets",
    "Mean signed offset",
    "Median signed offset",
    "Maximum signed offset",
    "P75 signed offset",
    "P90 signed offset",
    "offsets are above 1.5 seconds",
    "Filtered clean overlap",
    "Mean best B correction shift",
    "Median best B correction shift",
    "Mean estimated B lateness",
    "Median estimated B lateness",
    "Mean alignment score gain",
    "Median alignment score gain",
    "Mean number of bilateral alignment events",
    "Mean bilateral event coverage",
]:

    assert (
        required_text
        in combined_reference_text
    )


print("\n" + "=" * 88)
print("SELECTED NORMAL-ONLY REFERENCE TEXT READY")
print("=" * 88)

print(
    "Base reference metrics included:",
    len(
        normal_base_reference_values
    ),
)

print(
    "Global reference metrics included:",
    len(
        normal_global_reference_values
    ),
)

print(
    "No additional frozen statistics were exposed."
)

print(
    "No explicit anomaly profile or anomaly type was included."
)

normal_base_reference_text
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.
- Filtered clean overlap is around 6.25 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

normal_global_reference_text
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.

# Rationale-only full-semantics input and inference helpers

The evidence projection is unchanged. The output schema contains only `chain_of_thought` followed by `label`.


In [ ]:

# ============================================================
# SHARED RATIONALE-ONLY EXPERIMENT HELPERS
#
# The model-facing input projection is copied from the original
# binary consolidation notebook:
#   - same participation fields
#   - same filtered turns
#   - same local temporal fields
#   - same global temporal fields
#   - same coarse summaries
#   - same focused summaries
#
# The original binary-only source prompt is loaded from its exact saved
# prompt_template.txt file.
#
# The only model-facing output change is that the original label-only
# JSON block is replaced by chain_of_thought followed by label.
# No structured assessment fields are requested.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import copy
import hashlib
import json
import re
import time

import pandas as pd
import torch

from tqdm.auto import tqdm


import difflib

MAX_NEW_TOKENS_REASONING = 1024
MAX_NEW_TOKENS = MAX_NEW_TOKENS_REASONING

LABELS = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# EXACT SEMANTIC FIELDS PASSED TO THE MODEL
# ============================================================

COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEGMENT_MAP = {
    "segment_0_0_to_60_seconds": (
        "segment_0"
    ),

    "segment_1_60_to_120_seconds": (
        "segment_1"
    ),
}


# ============================================================
# EXACT TEMPORAL FIELDS PASSED TO THE MODEL
# ============================================================

LOCAL_FEATURE_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# FIELDS THAT MUST NEVER APPEAR IN THE MODEL PROMPT
# ============================================================

FORBIDDEN_PROMPT_KEYS = [
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "conversation_id",
    "participant_id",
    "metadata_path",

    "num_raw_vad_entries",

    "A_source_conversation",
    "B_source_conversation",

    "semantic_summary_source",
    "semantic_summary_coverage",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )


    missing_fields = [
        field

        for field in fields

        if field not in record
    ]


    assert not missing_fields, (
        f"Missing fields in {context}: "
        f"{missing_fields}"
    )


    return {
        field: copy.deepcopy(
            record[
                field
            ]
        )

        for field in fields
    }


# ============================================================
# TURN PROJECTION
#
# Database format:
#   {"start": 1.2, "end": 3.4}
#
# Prompt format:
#   [1.2, 3.4]
# ============================================================

def compact_turns(
    turns,
    context,
):
    assert isinstance(
        turns,
        list,
    ), (
        f"Expected list for {context}"
    )


    compact = []


    for index, turn in enumerate(
        turns
    ):

        assert isinstance(
            turn,
            dict,
        ), (
            f"Invalid turn in {context} "
            f"at index {index}"
        )


        assert "start" in turn
        assert "end" in turn


        start = float(
            turn[
                "start"
            ]
        )


        end = float(
            turn[
                "end"
            ]
        )


        assert end >= start


        compact.append([
            start,
            end,
        ])


    return compact


# ============================================================
# SEMANTIC INPUT PROJECTION
#
# Participant IDs and other metadata are deliberately excluded.
# ============================================================

def build_semantic_input(
    case,
):
    semantic_input = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        role_output = {}


        for (
            database_segment_name,
            prompt_segment_name,
        ) in SEGMENT_MAP.items():

            segment_record = (
                participant_semantics[
                    database_segment_name
                ]
            )


            coarse_summary = (
                select_exact_fields(
                    segment_record[
                        "coarse_summary"
                    ],

                    COARSE_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "coarse_summary"
                    ),
                )
            )


            focused_summary = (
                select_exact_fields(
                    segment_record[
                        "focused_summary"
                    ],

                    FOCUSED_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "focused_summary"
                    ),
                )
            )


            # Legacy focused-summary speaks must not exist.
            assert (
                "speaks"
                not in focused_summary
            )


            role_output[
                prompt_segment_name
            ] = {
                "coarse_summary": (
                    coarse_summary
                ),

                "focused_summary": (
                    focused_summary
                ),
            }


        semantic_input[
            role
        ] = role_output


    return semantic_input


# ============================================================
# BUILD THE EXACT MODEL INPUT
# ============================================================

def build_binary_model_input(
    case,
):
    local_features = (
        select_exact_fields(
            case[
                "local_temporal_features"
            ],

            LOCAL_FEATURE_FIELDS,

            "local_temporal_features",
        )
    )


    global_features_raw = (
        select_exact_fields(
            case[
                "global_shift_features"
            ],

            GLOBAL_FEATURE_FIELDS,

            "global_shift_features",
        )
    )


    # Stored in the database as a fraction.
    # Presented in the prompt as a percentage so that it is
    # directly comparable with the frozen NORMAL percentage.
    event_coverage_fraction = float(
        global_features_raw.pop(
            "best_event_coverage"
        )
    )


    global_features = {
        **global_features_raw,

        "best_event_coverage_percent": round(
            100.0
            * event_coverage_fraction,

            6,
        ),
    }


    payload = {
        "analysis_duration_seconds": float(
            case[
                "analysis_duration_seconds"
            ]
        ),

        "participant_A": {
            "speaks": bool(
                case[
                    "participant_A"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_A_filtered_turns"
                ],

                "participant_A_filtered_turns",
            ),
        },

        "participant_B": {
            "speaks": bool(
                case[
                    "participant_B"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_B_filtered_turns"
                ],

                "participant_B_filtered_turns",
            ),
        },

        "local_temporal_features": (
            local_features
        ),

        "global_shift_features": (
            global_features
        ),

        "semantic_summaries": (
            build_semantic_input(
                case
            )
        ),
    }


    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "participant_A",
        "participant_B",
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    }


    return payload


# ============================================================
# BUILD THE FINAL PROMPT FOR ONE CASE
# ============================================================

def build_binary_prompt_from_template(
    case,
    prompt_template,
):
    payload = build_binary_model_input(
        case
    )


    prompt = (
        prompt_template
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )


    # Strict leakage audit.
    prompt_lower = prompt.lower()


    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):

        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )


    return (
        prompt,
        payload,
    )


# ============================================================
# TEXT-ONLY QWEN INFERENCE
#
# The existing qwen_text_only helper used 700 output tokens.
# Here only 64 are needed because the expected output is:
#
#   {"label": "NORMAL"}
#
# or:
#
#   {"label": "ANOMALOUS"}
# ============================================================

def qwen_text_only_binary(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]


    inputs = (
        processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",

            processor_kwargs={
                "padding": True,
            },
        )
    )


    inputs = {
        key: (
            value.to(
                model.device
            )
            if hasattr(
                value,
                "to",
            )
            else value
        )

        for key, value
        in inputs.items()
    }


    input_token_count = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            use_cache=True,
        )


    generated_ids = output_ids[
        :,
        inputs[
            "input_ids"
        ].shape[1]:,
    ]


    raw_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    )


    return (
        raw_output,
        input_token_count,
    )


# ============================================================
# ROBUST JSON PARSING
# ============================================================

def extract_first_json_object(
    text,
):
    cleaned = text.strip()


    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    try:

        return json.loads(
            cleaned
        )

    except Exception:

        pass


    decoder = json.JSONDecoder()


    possible_starts = [
        index

        for index, character
        in enumerate(
            cleaned
        )

        if character == "{"
    ]


    for start in possible_starts:

        try:

            parsed, _ = (
                decoder.raw_decode(
                    cleaned[
                        start:
                    ]
                )
            )


            return parsed

        except Exception:

            continue


    return None


# ============================================================
# RATIONALE-ONLY OUTPUT SCHEMA
# ============================================================

RATIONALE_SCHEMA_KEYS = [
    "chain_of_thought",
    "label",
]


OLD_BINARY_OUTPUT_BLOCK = """
============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


RATIONALE_ONLY_OUTPUT_BLOCK = """
============================================================
FREE-FORM RATIONALE OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final binary label, provide one concise free-form
chain_of_thought rationale using at most 7 short numbered steps and no more
than 200 words in total.

Reason through the same evidence dimensions already defined above:

1. assess participation,
2. assess the local temporal evidence,
3. assess the global temporal evidence,
4. combine the local and global evidence using the existing temporal policy,
5. assess semantic compatibility,
6. identify which evidence dimension is decisive,
7. explain how the final binary label follows from the existing policy.

Explicitly discuss conflicting, weak, or limited evidence when present and
explain how it affects the decision.

Do not repeat all numerical feature values.
Do not restate the prompt or the complete evidence packet.
Include only evidence that directly supports the final binary decision.

The chain_of_thought must use only the evidence supplied in the prompt.
It must not introduce new evidence, thresholds, decision rules, labels,
anomaly subtypes, or delay magnitudes.

The chain_of_thought value must be one non-empty valid JSON string.
Do not use unescaped quotation marks or literal line breaks inside the
JSON string.

Do not output any structured assessment fields.
Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema and key order:

{{
  "chain_of_thought": "Free-form step-by-step rationale as one valid JSON string",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


def build_rationale_only_prompt_template(
    source_prompt_template,
):
    assert isinstance(
        source_prompt_template,
        str,
    )

    assert source_prompt_template.endswith(
        OLD_BINARY_OUTPUT_BLOCK
    ), (
        "The saved source prompt does not end with the exact "
        "original binary OUTPUT block."
    )

    rationale_prompt_template = (
        source_prompt_template[
            :-len(
                OLD_BINARY_OUTPUT_BLOCK
            )
        ]
        + RATIONALE_ONLY_OUTPUT_BLOCK
    )

    reconstructed_source_prompt = (
        rationale_prompt_template[
            :-len(
                RATIONALE_ONLY_OUTPUT_BLOCK
            )
        ]
        + OLD_BINARY_OUTPUT_BLOCK
    )

    assert (
        reconstructed_source_prompt
        == source_prompt_template
    ), (
        "Unexpected source-prompt modification."
    )

    return rationale_prompt_template


def parse_rationale_only_prediction(
    raw_output,
):
    parsed = extract_first_json_object(
        raw_output
    )

    chain_of_thought = None
    label = None
    schema_errors = []

    if isinstance(
        parsed,
        dict,
    ):

        if "chain_of_thought" in parsed:

            if isinstance(
                parsed[
                    "chain_of_thought"
                ],
                str,
            ):

                chain_of_thought = (
                    parsed[
                        "chain_of_thought"
                    ].strip()
                )

            else:

                schema_errors.append(
                    "chain_of_thought must be a JSON string."
                )

        if "label" in parsed:

            label = str(
                parsed[
                    "label"
                ]
            ).strip().upper()

        if not chain_of_thought:

            schema_errors.append(
                "chain_of_thought is empty or missing."
            )

        if label not in LABELS:

            schema_errors.append(
                f"label: {label}"
            )

        exact_key_order = (
            list(
                parsed.keys()
            )
            == RATIONALE_SCHEMA_KEYS
        )

        schema_exact = (
            exact_key_order
            and
            not schema_errors
        )

        if label in LABELS:

            return {
                "prediction": label,
                "chain_of_thought": (
                    chain_of_thought
                ),
                "parse_mode": (
                    "rationale_json"
                ),
                "schema_exact": bool(
                    schema_exact
                ),
                "parsed_output": parsed,
                "schema_errors": (
                    schema_errors
                ),
            }

    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )

    if plain_output in LABELS:

        return {
            "prediction": plain_output,
            "chain_of_thought": None,
            "parse_mode": (
                "exact_plaintext_fallback"
            ),
            "schema_exact": False,
            "parsed_output": None,
            "schema_errors": [
                "Rationale field missing."
            ],
        }

    return {
        "prediction": None,
        "chain_of_thought": (
            chain_of_thought
        ),
        "parse_mode": "invalid",
        "schema_exact": False,
        "parsed_output": parsed,
        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid rationale JSON prediction."
            ]
        ),
    }


# ============================================================
# FULL-SEMANTICS PAYLOAD AUDIT
# ============================================================

def collect_nested_keys_reasoning(
    value,
):
    keys = set()

    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in value.items():

            keys.add(
                str(key)
            )

            keys.update(
                collect_nested_keys_reasoning(
                    nested_value
                )
            )

    elif isinstance(
        value,
        list,
    ):

        for item in value:

            keys.update(
                collect_nested_keys_reasoning(
                    item
                )
            )

    return keys


FULL_SEMANTIC_REQUIRED_KEYS = [
    "semantic_summaries",
    "coarse_summary",
    "focused_summary",
    "speech_content_summary",
    "apparent_topic",
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


# ============================================================
# EXACT PROMPT CONFIGURATION
# ============================================================

def prepare_reasoning_experiment(
    *,
    experiment_version,
    source_experiment_name,
    experiment_title,
    required_source_markers,
    forbidden_source_markers,
    temporal_profiles_used,
    assessment_policy,
):
    source_experiment_dir = (
        OUT_DIR
        / source_experiment_name
    )

    source_prompt_path = (
        source_experiment_dir
        / "prompt_template.txt"
    )

    assert source_prompt_path.exists(), (
        "Saved source prompt not found:\n"
        f"{source_prompt_path}"
    )

    source_prompt_template = (
        source_prompt_path.read_text(
            encoding="utf-8"
        )
    )

    for marker in required_source_markers:

        assert marker in source_prompt_template, (
            "Required source-prompt marker missing: "
            f"{marker}"
        )

    for marker in forbidden_source_markers:

        assert marker not in source_prompt_template, (
            "Forbidden source-prompt marker found: "
            f"{marker}"
        )

    for semantic_marker in [
        "Coarse semantic information:",
        "Focused semantic information:",
        "- detailed_speech_summary",
        "- main_topic",
        "- secondary_topics",
        "- key_semantic_details",
    ]:

        assert semantic_marker in source_prompt_template, (
            "The source prompt is not a full-semantics prompt. "
            f"Missing: {semantic_marker}"
        )

    rationale_prompt_template = (
        build_rationale_only_prompt_template(
            source_prompt_template
        )
    )

    experiment_dir = (
        OUT_DIR
        / experiment_version
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),
        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),
        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),
        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),
        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),
        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),
        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),
        "rationale_outputs_csv": (
            experiment_dir
            / "rationale_outputs.csv"
        ),
        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),
        "source_prompt_copy": (
            experiment_dir
            / "source_prompt_template.txt"
        ),
        "prompt_diff": (
            experiment_dir
            / "prompt_diff_vs_source.txt"
        ),
        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }

    source_prompt_sha256 = sha256_text(
        source_prompt_template
    )

    rationale_prompt_sha256 = sha256_text(
        rationale_prompt_template
    )

    rationale_schema_sha256 = sha256_text(
        RATIONALE_ONLY_OUTPUT_BLOCK
    )

    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )

    paths[
        "prompt_template"
    ].write_text(
        rationale_prompt_template,
        encoding="utf-8",
    )

    paths[
        "source_prompt_copy"
    ].write_text(
        source_prompt_template,
        encoding="utf-8",
    )

    prompt_diff_lines = difflib.unified_diff(
        source_prompt_template.splitlines(),
        rationale_prompt_template.splitlines(),
        fromfile=(
            f"{source_experiment_name}/prompt_template.txt"
        ),
        tofile=(
            f"{experiment_version}/prompt_template.txt"
        ),
        lineterm="",
    )

    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(
            prompt_diff_lines
        ),
        encoding="utf-8",
    )

    example_prompt, example_payload = (
        build_binary_prompt_from_template(
            consolidation_cases[0],
            rationale_prompt_template,
        )
    )

    payload_keys = collect_nested_keys_reasoning(
        example_payload
    )

    for required_key in FULL_SEMANTIC_REQUIRED_KEYS:

        assert required_key in payload_keys, (
            "Full-semantic input field missing: "
            f"{required_key}"
        )

    output_tail = rationale_prompt_template[
        -len(
            RATIONALE_ONLY_OUTPUT_BLOCK
        ):
    ]

    assert (
        "FREE-FORM RATIONALE OUTPUT"
        in example_prompt
    )

    assert (
        '"chain_of_thought"'
        in output_tail
    )

    assert (
        '"label"'
        in output_tail
    )

    for structured_key in [
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
    ]:

        assert structured_key not in output_tail, (
            "Structured output field unexpectedly present: "
            f"{structured_key}"
        )

    assert (
        output_tail.find(
            '"chain_of_thought"'
        )
        <
        output_tail.find(
            '"label"'
        )
    )

    manifest = {
        "experiment_version": (
            experiment_version
        ),
        "experiment_title": (
            experiment_title
        ),
        "source_experiment": (
            source_experiment_name
        ),
        "source_prompt_path": str(
            source_prompt_path
        ),
        "source_prompt_sha256": (
            source_prompt_sha256
        ),
        "reasoning_prompt_sha256": (
            rationale_prompt_sha256
        ),
        "reasoning_schema_sha256": (
            rationale_schema_sha256
        ),
        "normal_reference_sha256": (
            normal_reference_sha256
        ),
        "semantic_input": (
            "coarse_and_focused"
        ),
        "focused_summaries_used": True,
        "temporal_profiles_used": (
            temporal_profiles_used
        ),
        "assessment_policy": (
            assessment_policy
        ),
        "structured_assessment_fields_used": False,
        "output_key_order": (
            RATIONALE_SCHEMA_KEYS
        ),
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "model_id": (
            MODEL_ID
        ),
        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },
        "only_prompt_change": (
            "The original label-only OUTPUT block is replaced "
            "by chain_of_thought followed by label. No structured "
            "assessment fields are requested."
        ),
        "all_model_facing_evidence_unchanged": True,
        "all_pre_output_prompt_text_unchanged": True,
    }

    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    config = {
        **manifest,
        "experiment_dir": (
            experiment_dir
        ),
        "source_prompt_template": (
            source_prompt_template
        ),
        "reasoning_prompt_template": (
            rationale_prompt_template
        ),
        "paths": paths,
    }

    print("=" * 88)
    print(
        f"{experiment_title} — CONFIGURATION READY"
    )
    print("=" * 88)

    print(
        "Experiment version:",
        experiment_version,
    )
    print(
        "Source experiment:",
        source_experiment_name,
    )
    print(
        "Source prompt SHA256:",
        source_prompt_sha256,
    )
    print(
        "Rationale prompt SHA256:",
        rationale_prompt_sha256,
    )
    print(
        "Semantic input:",
        "coarse_and_focused",
    )
    print(
        "Focused summaries used:",
        True,
    )
    print(
        "Temporal profiles:",
        temporal_profiles_used,
    )
    print(
        "Structured assessment fields used:",
        False,
    )
    print(
        "Output key order:",
        RATIONALE_SCHEMA_KEYS,
    )
    print(
        "Only source-prompt change:",
        "binary label-only output -> chain_of_thought then label",
    )
    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )
    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )
    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )

    return config


# ============================================================
# CHECKPOINTED INFERENCE
# ============================================================

def reasoning_utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def reasoning_atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )

    temporary_path = path.with_suffix(
        path.suffix
        + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def create_reasoning_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),
        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),
        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),
        "model_id": (
            MODEL_ID
        ),
        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),
        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),
        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),
        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),
        "semantic_input": (
            "coarse_and_focused"
        ),
        "focused_summaries_used": True,
        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),
        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),
        "structured_assessment_fields_used": False,
        "output_key_order": (
            RATIONALE_SCHEMA_KEYS
        ),
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "created_at_utc": (
            reasoning_utc_now()
        ),
        "updated_at_utc": (
            reasoning_utc_now()
        ),
        "records": {},
    }


def run_reasoning_experiment(
    config,
):
    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )

    if prediction_cache_path.exists():

        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )

        expected_cache = create_reasoning_cache(
            config
        )

        for key in [
            "experiment_version",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "structured_assessment_fields_used",
            "output_key_order",
            "max_new_tokens",
        ]:

            assert (
                prediction_cache[
                    key
                ]
                == expected_cache[
                    key
                ]
            ), (
                f"Cache mismatch for {key}"
            )

        print(
            "Resuming cache:",
            prediction_cache_path,
        )
        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )

    else:

        prediction_cache = (
            create_reasoning_cache(
                config
            )
        )

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

        print(
            "Created cache:",
            prediction_cache_path,
        )

    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert len(
        ordered_cases
    ) == 400

    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):

        case_id = str(
            case[
                "case_id"
            ]
        )

        prompt, model_input_payload = (
            build_binary_prompt_from_template(
                case,
                config[
                    "reasoning_prompt_template"
                ],
            )
        )

        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )

        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):

            assert required_key in (
                current_payload_keys
            ), (
                f"Missing full-semantic field "
                f"for case {case_id}: "
                f"{required_key}"
            )

        prompt_sha256 = sha256_text(
            prompt
        )

        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )

        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )

        if (
            existing_record is not None
            and
            existing_record.get(
                "prediction"
            ) in LABELS
        ):

            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )

            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )

            continue

        started = time.perf_counter()

        try:

            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )

            parsed_result = (
                parse_rationale_only_prediction(
                    raw_output
                )
            )

            generation_error = None

        except Exception as exc:

            raw_output = ""
            input_token_count = None

            parsed_result = {
                "prediction": None,
                "chain_of_thought": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
            }

            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )

            if torch.cuda.is_available():

                torch.cuda.empty_cache()

        elapsed_seconds = (
            time.perf_counter()
            - started
        )

        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": case_id,
            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),
            "case_family": (
                get_case_family(
                    case
                )
            ),
            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),
            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),
            "prompt_sha256": (
                prompt_sha256
            ),
            "input_payload_sha256": (
                input_payload_sha256
            ),
            "semantic_input": (
                "coarse_and_focused"
            ),
            "focused_summaries_used": True,
            "input_token_count": (
                input_token_count
            ),
            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),
            "raw_output": (
                raw_output
            ),
            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),
            "chain_of_thought": (
                parsed_result[
                    "chain_of_thought"
                ]
            ),
            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),
            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),
            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),
            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),
            "generation_error": (
                generation_error
            ),
            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),
            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }

        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)
    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )
    print(
        "Prediction cache:",
        prediction_cache_path,
    )

    return prediction_cache


# ============================================================
# EVALUATION
# ============================================================

def count_rationale_words(
    text,
):
    return len(
        re.findall(
            r"\b[\w'-]+\b",
            str(
                text
            ),
        )
    )


def count_rationale_steps(
    text,
):
    return len(
        re.findall(
            r"(?:^|\s)[1-7][\.\)]",
            str(
                text
            ),
        )
    )


def summarize_reasoning_group(
    group,
):
    valid_group = group[
        group[
            "valid_prediction"
        ]
    ]

    return {
        "total_cases": int(
            len(
                group
            )
        ),
        "valid_predictions": int(
            len(
                valid_group
            )
        ),
        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),
        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),
        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),
        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),
        "accuracy_on_valid": (
            float(
                (
                    valid_group[
                        "gold_label"
                    ]
                    ==
                    valid_group[
                        "prediction"
                    ]
                ).mean()
            )
            if len(
                valid_group
            )
            else float(
                "nan"
            )
        ),
        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),
        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
        "mean_rationale_words": float(
            group[
                "rationale_word_count"
            ].mean()
        ),
    }


def evaluate_reasoning_experiment(
    config,
):
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        matthews_corrcoef,
        precision_score,
        recall_score,
    )

    from IPython.display import display

    prediction_cache = json.loads(
        config[
            "paths"
        ][
            "prediction_cache"
        ].read_text(
            encoding="utf-8"
        )
    )

    assert (
        prediction_cache[
            "experiment_version"
        ]
        == config[
            "experiment_version"
        ]
    )

    assert (
        prediction_cache[
            "reasoning_prompt_sha256"
        ]
        == config[
            "reasoning_prompt_sha256"
        ]
    )

    result_rows = []

    for case in consolidation_cases:

        case_id = str(
            case[
                "case_id"
            ]
        )

        record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )

        result_rows.append({
            "case_id": case_id,
            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),
            "case_family": (
                get_case_family(
                    case
                )
            ),
            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),
            "gold_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),
            "prediction": (
                None
                if record is None
                else record.get(
                    "prediction"
                )
            ),
            "chain_of_thought": (
                None
                if record is None
                else record.get(
                    "chain_of_thought"
                )
            ),
            "parse_mode": (
                "missing"
                if record is None
                else record.get(
                    "parse_mode"
                )
            ),
            "schema_exact": (
                False
                if record is None
                else bool(
                    record.get(
                        "schema_exact",
                        False,
                    )
                )
            ),
            "schema_errors": (
                None
                if record is None
                else json.dumps(
                    record.get(
                        "schema_errors",
                        [],
                    ),
                    ensure_ascii=False,
                )
            ),
            "input_token_count": (
                None
                if record is None
                else record.get(
                    "input_token_count"
                )
            ),
            "elapsed_seconds": (
                None
                if record is None
                else record.get(
                    "elapsed_seconds"
                )
            ),
            "raw_output": (
                ""
                if record is None
                else record.get(
                    "raw_output",
                    "",
                )
            ),
            "generation_error": (
                None
                if record is None
                else record.get(
                    "generation_error"
                )
            ),
        })

    results_df = pd.DataFrame(
        result_rows
    )

    results_df[
        "valid_prediction"
    ] = results_df[
        "prediction"
    ].isin(
        LABELS
    )

    results_df[
        "correct"
    ] = (
        results_df[
            "valid_prediction"
        ]
        &
        (
            results_df[
                "gold_label"
            ]
            ==
            results_df[
                "prediction"
            ]
        )
    )

    results_df[
        "rationale_present"
    ] = (
        results_df[
            "chain_of_thought"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    results_df[
        "rationale_word_count"
    ] = (
        results_df[
            "chain_of_thought"
        ]
        .fillna("")
        .apply(
            count_rationale_words
        )
    )

    results_df[
        "rationale_step_count"
    ] = (
        results_df[
            "chain_of_thought"
        ]
        .fillna("")
        .apply(
            count_rationale_steps
        )
    )

    results_df[
        "rationale_mentions_prediction"
    ] = results_df.apply(
        lambda row: (
            str(
                row[
                    "prediction"
                ]
            ).lower()
            in
            str(
                row[
                    "chain_of_thought"
                ]
            ).lower()
        )
        if row[
            "prediction"
        ] in LABELS
        else False,
        axis=1,
    )

    assert len(
        results_df
    ) == 400

    assert results_df[
        "case_id"
    ].is_unique

    valid_df = results_df[
        results_df[
            "valid_prediction"
        ]
    ].copy()

    invalid_df = results_df[
        ~results_df[
            "valid_prediction"
        ]
    ].copy()

    assert len(
        valid_df
    ) > 0

    y_true = valid_df[
        "gold_label"
    ]

    y_pred = valid_df[
        "prediction"
    ]

    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            "NORMAL",
            "ANOMALOUS",
        ],
    )

    confusion_df = pd.DataFrame(
        confusion,
        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],
        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )

    true_normal = int(
        confusion[0, 0]
    )
    false_anomalous = int(
        confusion[0, 1]
    )

    normal_recall = (
        true_normal
        /
        (
            true_normal
            +
            false_anomalous
        )
        if (
            true_normal
            +
            false_anomalous
        )
        else float(
            "nan"
        )
    )

    metrics = {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),
        "total_cases": int(
            len(
                results_df
            )
        ),
        "valid_predictions": int(
            len(
                valid_df
            )
        ),
        "invalid_predictions": int(
            len(
                invalid_df
            )
        ),
        "exact_json_schema_rate": float(
            results_df[
                "schema_exact"
            ].mean()
        ),
        "rationale_present_rate": float(
            results_df[
                "rationale_present"
            ].mean()
        ),
        "mean_rationale_words": float(
            results_df[
                "rationale_word_count"
            ].mean()
        ),
        "mean_rationale_steps": float(
            results_df[
                "rationale_step_count"
            ].mean()
        ),
        "rationale_mentions_prediction_rate": float(
            results_df[
                "rationale_mentions_prediction"
            ].mean()
        ),
        "accuracy_valid_predictions": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "strict_accuracy_invalid_as_wrong": float(
            results_df[
                "correct"
            ].mean()
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "anomalous_precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),
        "anomalous_recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),
        "anomalous_f1": float(
            f1_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),
        "normal_recall_specificity": float(
            normal_recall
        ),
        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                y_true,
                y_pred,
            )
        ),
        "confusion_matrix_label_order": [
            "NORMAL",
            "ANOMALOUS",
        ],
        "confusion_matrix": (
            confusion.tolist()
        ),
    }

    family_rows = []

    for family_name, family_group in results_df.groupby(
        "case_family",
        sort=True,
    ):

        family_rows.append({
            "case_family": family_name,
            **summarize_reasoning_group(
                family_group
            ),
        })

    family_metrics_df = pd.DataFrame(
        family_rows
    )

    variant_rows = []

    for variant_name, variant_group in results_df.groupby(
        "case_variant",
        sort=True,
    ):

        variant_rows.append({
            "case_variant": variant_name,
            **summarize_reasoning_group(
                variant_group
            ),
        })

    variant_metrics_df = pd.DataFrame(
        variant_rows
    )

    classification_report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=[
                "NORMAL",
                "ANOMALOUS",
            ],
            output_dict=True,
            zero_division=0,
        )
    ).T

    error_df = results_df[
        (
            ~results_df[
                "valid_prediction"
            ]
        )
        |
        (
            results_df[
                "gold_label"
            ]
            !=
            results_df[
                "prediction"
            ]
        )
    ].copy()

    source_group_rows = []

    for source_group_id, source_group in results_df.groupby(
        "source_group_id"
    ):

        source_group_rows.append({
            "source_group_id": (
                source_group_id
            ),
            "num_cases": int(
                len(
                    source_group
                )
            ),
            "all_predictions_valid": bool(
                source_group[
                    "valid_prediction"
                ].all()
            ),
            "all_cases_correct": bool(
                source_group[
                    "valid_prediction"
                ].all()
                and
                source_group[
                    "correct"
                ].all()
            ),
        })

    source_group_df = pd.DataFrame(
        source_group_rows
    )

    assert len(
        source_group_df
    ) == 100

    assert set(
        source_group_df[
            "num_cases"
        ]
    ) == {
        4
    }

    metrics[
        "source_group_exact_match_rate"
    ] = float(
        source_group_df[
            "all_cases_correct"
        ].mean()
    )

    paths = config[
        "paths"
    ]

    results_df.to_csv(
        paths[
            "predictions_csv"
        ],
        index=False,
    )

    results_df[
        [
            "case_id",
            "source_group_id",
            "case_family",
            "case_variant",
            "gold_label",
            "prediction",
            "chain_of_thought",
            "rationale_word_count",
            "rationale_step_count",
            "rationale_mentions_prediction",
            "schema_exact",
            "correct",
        ]
    ].to_csv(
        paths[
            "rationale_outputs_csv"
        ],
        index=False,
    )

    confusion_df.to_csv(
        paths[
            "confusion_matrix_csv"
        ]
    )

    family_metrics_df.to_csv(
        paths[
            "family_metrics_csv"
        ],
        index=False,
    )

    variant_metrics_df.to_csv(
        paths[
            "variant_metrics_csv"
        ],
        index=False,
    )

    error_df.to_csv(
        paths[
            "errors_csv"
        ],
        index=False,
    )

    paths[
        "metrics_json"
    ].write_text(
        json.dumps(
            metrics,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print("=" * 88)
    print(
        f"{config['experiment_title']} — RESULTS"
    )
    print("=" * 88)
    print(
        "Total cases:",
        metrics[
            "total_cases"
        ],
    )
    print(
        "Valid predictions:",
        metrics[
            "valid_predictions"
        ],
    )
    print(
        "Invalid predictions:",
        metrics[
            "invalid_predictions"
        ],
    )
    print(
        "Exact rationale-schema rate:",
        f'{metrics["exact_json_schema_rate"]:.4f}',
    )
    print(
        "Rationale present rate:",
        f'{metrics["rationale_present_rate"]:.4f}',
    )
    print(
        "Mean rationale words:",
        f'{metrics["mean_rationale_words"]:.2f}',
    )
    print(
        "Mean rationale steps:",
        f'{metrics["mean_rationale_steps"]:.2f}',
    )
    print(
        "Accuracy:",
        f'{metrics["accuracy_valid_predictions"]:.4f}',
    )
    print(
        "Balanced accuracy:",
        f'{metrics["balanced_accuracy"]:.4f}',
    )
    print(
        "ANOMALOUS precision:",
        f'{metrics["anomalous_precision"]:.4f}',
    )
    print(
        "ANOMALOUS recall:",
        f'{metrics["anomalous_recall"]:.4f}',
    )
    print(
        "ANOMALOUS F1:",
        f'{metrics["anomalous_f1"]:.4f}',
    )
    print(
        "NORMAL recall / specificity:",
        f'{metrics["normal_recall_specificity"]:.4f}',
    )
    print(
        "MCC:",
        f'{metrics["matthews_correlation_coefficient"]:.4f}',
    )
    print(
        "Matched source-group exact rate:",
        f'{metrics["source_group_exact_match_rate"]:.4f}',
    )

    print("\nCONFUSION MATRIX")
    display(
        confusion_df
    )

    print("\nCLASSIFICATION REPORT")
    display(
        classification_report_df
    )

    print("\nPER CASE FAMILY")
    display(
        family_metrics_df
    )

    print("\nPER EXACT CASE VARIANT")
    display(
        variant_metrics_df
    )

    print(
        "\nSaved cache:",
        paths[
            "prediction_cache"
        ],
    )
    print(
        "Saved prompt:",
        paths[
            "prompt_template"
        ],
    )
    print(
        "Saved prompt diff:",
        paths[
            "prompt_diff"
        ],
    )
    print(
        "Saved predictions:",
        paths[
            "predictions_csv"
        ],
    )
    print(
        "Saved rationale outputs:",
        paths[
            "rationale_outputs_csv"
        ],
    )
    print(
        "Saved errors:",
        paths[
            "errors_csv"
        ],
    )

    if len(
        invalid_df
    ):

        print(
            "\nWARNING: Invalid predictions were excluded "
            "from the confusion matrix."
        )

        display(
            invalid_df[
                [
                    "case_id",
                    "case_family",
                    "case_variant",
                    "parse_mode",
                    "raw_output",
                    "generation_error",
                ]
            ]
        )

    else:

        print(
            "\nAll 400 cases produced valid "
            "binary predictions."
        )

    return {
        "metrics": metrics,
        "results_df": results_df,
        "confusion_df": confusion_df,
        "family_metrics_df": family_metrics_df,
        "variant_metrics_df": variant_metrics_df,
        "classification_report_df": (
            classification_report_df
        ),
        "error_df": error_df,
        "source_group_df": source_group_df,
    }


# R1 — Full Semantics, NORMAL References Only, Rationale → Label

**Controlled ablation:** all evidence, frozen references, decision rules, model settings, and decoding settings remain unchanged. The six structured diagnostic fields are removed, and the model returns only a concise free-form rationale followed by the binary label.


In [ ]:
# ============================================================
# R1 CONFIGURATION
# FULL SEMANTICS + NORMAL REFERENCES ONLY
# FREE-FORM CHAIN_OF_THOUGHT → LABEL
# NO STRUCTURED ASSESSMENT FIELDS
#
# Exact source:
#   Experiment 2 — Independent Normality Requirements
#
# Only model-facing output change relative to Binary Only:
#   replace label-only output with chain_of_thought then label
# ============================================================

R1_CONFIG = prepare_reasoning_experiment(
    experiment_version=(
        "reasoning_r1_full_semantics_"
        "normal_references_only_"
        "chain_of_thought_then_label_"
        "no_structured_fields"
    ),

    source_experiment_name=(
        "binary_only_consolidation_"
        "normal_definition_v2"
    ),

    experiment_title=(
        "R1 — Full Semantics, No LAG Profiles, "
        "Free-Form Rationale Then Label"
    ),

    required_source_markers=[
        "NORMAL requires three independent properties",
        "FROZEN NORMAL LOCAL-TIMING REFERENCE",
        "FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE",
        "Coarse semantic information:",
        "Focused semantic information:",
    ],

    forbidden_source_markers=[
        "FROZEN NON-NORMAL LOCAL-TIMING REFERENCE PROFILES",
        "Frozen LAG_2 local reference pattern",
        "Frozen LAG_3 local reference pattern",
        "TEMPORAL PROFILE COMPARISON",
        "ORDERED AND INDEPENDENT EVIDENCE ASSESSMENT",
    ],

    temporal_profiles_used=[
        "NORMAL",
    ],

    assessment_policy=(
        "Original Experiment 2 independent-normality "
        "requirements, unchanged."
    ),
)


R1 — Full Semantics, No LAG Profiles, Free-Form Rationale Then Label — CONFIGURATION READY
Experiment version: reasoning_r1_full_semantics_normal_references_only_chain_of_thought_then_label_no_structured_fields
Source experiment: binary_only_consolidation_normal_definition_v2
Source prompt SHA256: ed2d68e14f1fa667650b11ef8e2c75a7f03c48c7e3faa1dfd0dabecaa14bbace
Rationale prompt SHA256: 6a23f9dc1427f09eabe7f95fcc4a997581df9c52754dc872759642762dc6e531
Semantic input: coarse_and_focused
Focused summaries used: True
Temporal profiles: ['NORMAL']
Structured assessment fields used: False
Output key order: ['chain_of_thought', 'label']
Only source-prompt change: binary label-only output -> chain_of_thought then label
Example prompt characters: 25392
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought_then_label_no_structured_fields/predictions_cache.json
Existing cache: True


In [ ]:
# ============================================================
# RUN OR RESUME RATIONALE-ONLY EXPERIMENT
# ============================================================

R1_CACHE = run_reasoning_experiment(
    R1_CONFIG
)


Resuming cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought_then_label_no_structured_fields/predictions_cache.json
Existing records: 400


reasoning_r1_full_semantics_normal_references_only_chain_of_thought_then_label_no_structured_fields:   0%|    …


R1 — Full Semantics, No LAG Profiles, Free-Form Rationale Then Label — INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought_then_label_no_structured_fields/predictions_cache.json


## Verify that the rationale was emitted before the label

This audit reads the new experiment cache and verifies the raw JSON key order. It does not rerun the model.


In [ ]:
# ============================================================
# OUTPUT-ORDER AND COMPLETENESS AUDIT
# ============================================================

from pathlib import Path
import json
import pandas as pd
from IPython.display import display


CACHE_PATH = R1_CONFIG[
    "paths"
][
    "prediction_cache"
]


assert CACHE_PATH.exists(), (
    f"Prediction cache not found: {CACHE_PATH}"
)


cache_data = json.loads(
    CACHE_PATH.read_text(
        encoding="utf-8"
    )
)


records = cache_data.get(
    "records",
    {},
)


if isinstance(
    records,
    dict,
):

    records = list(
        records.values()
    )


cache_df = pd.DataFrame(
    records
)


print(
    "Generated samples:",
    len(
        cache_df
    ),
)


def key_position(
    raw_output,
    key,
):
    return str(
        raw_output
    ).find(
        f'"{key}"'
    )


cache_df[
    "rationale_position"
] = cache_df[
    "raw_output"
].apply(
    lambda text: key_position(
        text,
        "chain_of_thought",
    )
)


cache_df[
    "label_position"
] = cache_df[
    "raw_output"
].apply(
    lambda text: key_position(
        text,
        "label",
    )
)


cache_df[
    "has_chain_of_thought"
] = (
    cache_df[
        "rationale_position"
    ]
    >= 0
)


cache_df[
    "has_label"
] = (
    cache_df[
        "label_position"
    ]
    >= 0
)


cache_df[
    "rationale_before_label"
] = (
    cache_df[
        "has_chain_of_thought"
    ]
    &
    cache_df[
        "has_label"
    ]
    &
    (
        cache_df[
            "rationale_position"
        ]
        <
        cache_df[
            "label_position"
        ]
    )
)


structured_keys = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


cache_df[
    "contains_structured_field"
] = cache_df[
    "raw_output"
].fillna("").astype(str).apply(
    lambda text: any(
        f'"{key}"'
        in text
        for key in structured_keys
    )
)


print("\nPARSE MODES")

display(
    cache_df[
        "parse_mode"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "parse_mode"
    )
    .reset_index(
        name="count"
    )
)


print("\nOUTPUT ORDER SUMMARY")

display(
    cache_df[
        "rationale_before_label"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "rationale_before_label"
    )
    .reset_index(
        name="count"
    )
)


print("\nOUTPUT COMPLETENESS")

display(
    cache_df[
        [
            "case_id",
            "case_family",
            "parse_mode",
            "has_chain_of_thought",
            "has_label",
            "rationale_before_label",
            "contains_structured_field",
        ]
    ]
)


if len(
    cache_df
):

    assert cache_df[
        "rationale_before_label"
    ].all(), (
        "At least one output did not emit chain_of_thought "
        "before label. Inspect those rows before evaluation."
    )

    assert not cache_df[
        "contains_structured_field"
    ].any(), (
        "At least one output unexpectedly contains a structured "
        "assessment field."
    )


Generated samples: 400

PARSE MODES


,parse_mode,count
0,rationale_json,400



OUTPUT ORDER SUMMARY


,rationale_before_label,count
0,True,400



OUTPUT COMPLETENESS


,case_id,case_family,parse_mode,has_chain_of_thought,has_label,rationale_before_label,contains_structured_field
0,consolidation_lag_2sec_000,lag,rationale_json,True,True,True,False
1,consolidation_lag_2sec_001,lag,rationale_json,True,True,True,False
2,consolidation_lag_2sec_003,lag,rationale_json,True,True,True,False
3,consolidation_lag_2sec_007,lag,rationale_json,True,True,True,False
4,consolidation_lag_2sec_009,lag,rationale_json,True,True,True,False
...,...,...,...,...,...,...,...
395,consolidation_wrong_partner_095,wrong_partner,rationale_json,True,True,True,False
396,consolidation_wrong_partner_096,wrong_partner,rationale_json,True,True,True,False
397,consolidation_wrong_partner_097,wrong_partner,rationale_json,True,True,True,False
398,consolidation_wrong_partner_098,wrong_partner,rationale_json,True,True,True,False


## Evaluate the rationale-only experiment

This evaluates the cached predictions, saves all metrics and outputs, and reports rationale-format diagnostics.


In [ ]:
# ============================================================
# EVALUATE RATIONALE-ONLY EXPERIMENT
# ============================================================

R1_EVALUATION = evaluate_reasoning_experiment(
    R1_CONFIG
)


R1 — Full Semantics, No LAG Profiles, Free-Form Rationale Then Label — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact rationale-schema rate: 1.0000
Rationale present rate: 1.0000
Mean rationale words: 168.61
Mean rationale steps: 7.78
Accuracy: 0.6975
Balanced accuracy: 0.7650
ANOMALOUS precision: 0.9497
ANOMALOUS recall: 0.6300
ANOMALOUS F1: 0.7575
NORMAL recall / specificity: 0.9000
MCC: 0.4590
Matched source-group exact rate: 0.1800

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,90,10
Gold ANOMALOUS,111,189



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.447761,0.9000,0.598007,100.0000
ANOMALOUS,0.949749,0.6300,0.757515,300.0000
accuracy,0.697500,0.6975,0.697500,0.6975
macro avg,0.698755,0.7650,0.677761,400.0000
weighted avg,0.824252,0.6975,0.717638,400.0000



PER CASE FAMILY


,case_family,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate,mean_rationale_words
0,lag,100,100,0,57,43,43,0.43,0.43,1.0,184.21
1,normal,100,100,0,90,10,90,0.90,0.90,1.0,150.50
2,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0,168.67
3,wrong_partner,100,100,0,54,46,46,0.46,0.46,1.0,171.07



PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate,mean_rationale_words
0,lag_2sec,50,50,0,32,18,18,0.36,0.36,1.0,182.00
1,lag_3sec,50,50,0,25,25,25,0.50,0.50,1.0,186.42
2,normal,100,100,0,90,10,90,0.90,0.90,1.0,150.50
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0,168.67
4,wrong_partner,100,100,0,54,46,46,0.46,0.46,1.0,171.07



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought_then_label_no_structured_fields/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought_then_label_no_structured_fields/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought_then_label_no_structured_fields/prompt_diff_vs_source.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought_then_label_no_structured_fields/predictions_all_400.csv
Saved rationale outputs: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought_then_label_no

In [ ]:
# ============================================================
# PREPARE PER-SUBSET RATIONALE-ONLY ANALYSIS
# WITH EXACT TEMPORAL FEATURES FOR EVERY PRINTED TRACE
# ============================================================

import json
import re

import pandas as pd
from IPython.display import display


# ============================================================
# REQUIRED EXPERIMENT OBJECTS
# ============================================================

assert "R1_EVALUATION" in globals(), (
    "R1_EVALUATION was not found. "
    "Run the evaluation cell first."
)


assert "consolidation_cases" in globals(), (
    "consolidation_cases was not found. "
    "Run the final-database loading cell first."
)


assert "build_binary_model_input" in globals(), (
    "build_binary_model_input was not found. "
    "Run the model-input construction cells first."
)


# ============================================================
# LOAD RESULTS DATAFRAME
# ============================================================

analysis_df = (
    R1_EVALUATION[
        "results_df"
    ]
    .copy()
)


REQUIRED_COLUMNS = [
    "case_id",
    "case_family",
    "case_variant",
    "gold_label",
    "prediction",
    "chain_of_thought",
]


missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in analysis_df.columns
]


assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)


assert len(
    analysis_df
) == 400, (
    f"Expected 400 cases, found {len(analysis_df)}."
)


# ============================================================
# BUILD CASE LOOKUP FROM THE ORIGINAL 400-CASE DATABASE
# ============================================================

case_lookup = {
    str(
        case[
            "case_id"
        ]
    ): case

    for case in consolidation_cases
}


assert len(
    case_lookup
) == 400, (
    "Expected 400 unique cases in the consolidation database, "
    f"found {len(case_lookup)}."
)


missing_case_ids = [
    str(
        case_id
    )
    for case_id in analysis_df[
        "case_id"
    ]
    if str(
        case_id
    ) not in case_lookup
]


assert not missing_case_ids, (
    "Some evaluation cases were not found in the original database: "
    f"{missing_case_ids[:10]}"
)


# ============================================================
# NORMALISE RESULT COLUMNS
# ============================================================

for column in [
    "gold_label",
    "prediction",
]:

    analysis_df[
        column
    ] = (
        analysis_df[
            column
        ]
        .fillna(
            "MISSING"
        )
        .astype(str)
        .str.strip()
        .str.upper()
    )


analysis_df[
    "case_id"
] = (
    analysis_df[
        "case_id"
    ]
    .astype(str)
    .str.strip()
)


analysis_df[
    "case_family"
] = (
    analysis_df[
        "case_family"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


analysis_df[
    "case_variant"
] = (
    analysis_df[
        "case_variant"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


analysis_df[
    "chain_of_thought"
] = (
    analysis_df[
        "chain_of_thought"
    ]
    .fillna("")
    .astype(str)
)


# ============================================================
# RATIONALE DIAGNOSTICS
# ============================================================

def count_words(
    text,
):

    return len(
        re.findall(
            r"\b[\w'-]+\b",
            str(
                text
            ),
        )
    )


def count_numbered_steps(
    text,
):

    # Counts numbered-list markers such as:
    # "1. Participation" or "2) Temporal evidence"
    #
    # The required whitespace after "." or ")" prevents decimal
    # values such as "1.5 seconds" from being counted as steps.

    return len(
        re.findall(
            r"(?:^|\s)\d+[\.\)]\s+",
            str(
                text
            ),
        )
    )


analysis_df[
    "reasoning_word_count"
] = (
    analysis_df[
        "chain_of_thought"
    ]
    .apply(
        count_words
    )
)


analysis_df[
    "reasoning_step_count"
] = (
    analysis_df[
        "chain_of_thought"
    ]
    .apply(
        count_numbered_steps
    )
)


lower_reasoning = (
    analysis_df[
        "chain_of_thought"
    ]
    .str.lower()
)


analysis_df[
    "mentions_participation"
] = lower_reasoning.str.contains(
    "participation",
    regex=False,
)


analysis_df[
    "mentions_local"
] = lower_reasoning.str.contains(
    "local",
    regex=False,
)


analysis_df[
    "mentions_global"
] = lower_reasoning.str.contains(
    "global",
    regex=False,
)


analysis_df[
    "mentions_temporal"
] = lower_reasoning.str.contains(
    "temporal",
    regex=False,
)


analysis_df[
    "mentions_semantic"
] = lower_reasoning.str.contains(
    "semantic",
    regex=False,
)


analysis_df[
    "mentions_limited_or_weak"
] = lower_reasoning.str.contains(
    r"\b("
    r"limited|"
    r"weak|"
    r"uncertain|"
    r"insufficient|"
    r"unreliable"
    r")\b",
    regex=True,
)


analysis_df[
    "mentions_predicted_label"
] = analysis_df.apply(
    lambda row: (
        row[
            "prediction"
        ].lower()
        in
        row[
            "chain_of_thought"
        ].lower()
    )
    if row[
        "prediction"
    ] in [
        "NORMAL",
        "ANOMALOUS",
    ]
    else False,
    axis=1,
)


# ============================================================
# SELECT REPRESENTATIVE CASES
# ============================================================

def select_representative_cases(
    subset_df,
    n=10,
):

    if len(
        subset_df
    ) == 0:

        return subset_df.copy()


    work_df = subset_df.copy()


    number_to_select = min(
        n,
        len(
            work_df
        ),
    )


    work_df = (
        work_df
        .sort_values(
            [
                "case_variant",
                "reasoning_word_count",
                "case_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    if number_to_select == 1:

        positions = [
            0
        ]

    else:

        positions = [
            round(
                index
                *
                (
                    len(
                        work_df
                    )
                    - 1
                )
                /
                (
                    number_to_select
                    - 1
                )
            )

            for index in range(
                number_to_select
            )
        ]


    positions = sorted(
        set(
            positions
        )
    )


    return (
        work_df
        .iloc[
            positions
        ]
        .head(
            number_to_select
        )
    )


# ============================================================
# GET EXACT MODEL-FACING TEMPORAL FEATURES
# ============================================================

def get_exact_temporal_features(
    case_id,
):

    case_id = str(
        case_id
    )


    assert case_id in case_lookup, (
        f"Case not found in original database: {case_id}"
    )


    original_case = case_lookup[
        case_id
    ]


    # This reconstructs the exact evidence packet that was passed
    # to the model during inference.
    model_payload = build_binary_model_input(
        original_case
    )


    assert (
        "local_temporal_features"
        in model_payload
    )


    assert (
        "global_shift_features"
        in model_payload
    )


    local_temporal_features = (
        model_payload[
            "local_temporal_features"
        ]
    )


    global_shift_features = (
        model_payload[
            "global_shift_features"
        ]
    )


    return {
        "local_temporal_features": (
            local_temporal_features
        ),

        "global_shift_features": (
            global_shift_features
        ),
    }


# ============================================================
# PRINT REPRESENTATIVE RATIONALES
# WITH TEMPORAL FEATURES BEFORE EACH RATIONALE
# ============================================================

def print_reasoning_traces(
    subset_df,
    n=10,
):

    selected_df = select_representative_cases(
        subset_df,
        n=n,
    )


    print(
        "\nREPRESENTATIVE MODEL-GENERATED RATIONALES:",
        len(
            selected_df
        ),
    )


    for trace_index, (_, row) in enumerate(
        selected_df.iterrows(),
        start=1,
    ):

        case_id = str(
            row[
                "case_id"
            ]
        )


        temporal_features = get_exact_temporal_features(
            case_id
        )


        local_temporal_features = (
            temporal_features[
                "local_temporal_features"
            ]
        )


        global_shift_features = (
            temporal_features[
                "global_shift_features"
            ]
        )


        print(
            "\n"
            +
            "=" * 110
        )

        print(
            f"TRACE {trace_index}"
        )

        print(
            "=" * 110
        )


        print(
            "CASE ID:",
            case_id,
        )

        print(
            "CASE FAMILY:",
            row[
                "case_family"
            ],
        )

        print(
            "CASE VARIANT:",
            row[
                "case_variant"
            ],
        )

        print(
            "GOLD LABEL:",
            row[
                "gold_label"
            ],
        )

        print(
            "PREDICTION:",
            row[
                "prediction"
            ],
        )


        # ====================================================
        # LOCAL TEMPORAL FEATURES
        # ====================================================

        print(
            "\nLOCAL TEMPORAL FEATURES "
            "(EXACT MODEL INPUT)"
        )

        print(
            json.dumps(
                local_temporal_features,
                indent=2,
                ensure_ascii=False,
            )
        )


        # ====================================================
        # GLOBAL TEMPORAL FEATURES
        # ====================================================

        print(
            "\nGLOBAL SHIFT FEATURES "
            "(EXACT MODEL INPUT)"
        )

        print(
            json.dumps(
                global_shift_features,
                indent=2,
                ensure_ascii=False,
            )
        )


        # ====================================================
        # RATIONALE LENGTH
        # ====================================================

        print(
            "\nRATIONALE LENGTH"
        )

        print(
            "Words:",
            row[
                "reasoning_word_count"
            ],
        )

        print(
            "Detected numbered steps:",
            row[
                "reasoning_step_count"
            ],
        )


        # ====================================================
        # GENERATED RATIONALE
        # ====================================================

        print(
            "\nMODEL-GENERATED RATIONALE"
        )

        print(
            row[
                "chain_of_thought"
            ]
        )


# ============================================================
# ANALYSE ONE SUBSET
# ============================================================

def analyse_subset(
    subset_df,
    title,
    n_traces=10,
):

    subset_df = subset_df.copy()


    print(
        "\n"
        +
        "#" * 110
    )

    print(
        title
    )

    print(
        "#" * 110
    )

    print(
        "Number of cases:",
        len(
            subset_df
        ),
    )


    if len(
        subset_df
    ) == 0:

        print(
            "No cases found in this subset."
        )

        return


    # ========================================================
    # CASE-VARIANT DISTRIBUTION
    # ========================================================

    print(
        "\nCASE VARIANT DISTRIBUTION"
    )

    display(
        subset_df[
            "case_variant"
        ]
        .value_counts(
            dropna=False
        )
        .rename_axis(
            "case_variant"
        )
        .reset_index(
            name="count"
        )
    )


    # ========================================================
    # RATIONALE LENGTH
    # ========================================================

    print(
        "\nRATIONALE LENGTH STATISTICS"
    )

    display(
        subset_df[
            [
                "reasoning_word_count",
                "reasoning_step_count",
            ]
        ]
        .describe()
        .round(
            2
        )
    )


    # ========================================================
    # RATIONALE CONTENT AUDIT
    # ========================================================

    audit_rows = []


    for column, description in [
        (
            "mentions_participation",
            "Mentions participation",
        ),
        (
            "mentions_local",
            "Mentions local evidence",
        ),
        (
            "mentions_global",
            "Mentions global evidence",
        ),
        (
            "mentions_temporal",
            "Mentions temporal evidence",
        ),
        (
            "mentions_semantic",
            "Mentions semantic evidence",
        ),
        (
            "mentions_limited_or_weak",
            "Mentions weak/limited evidence",
        ),
        (
            "mentions_predicted_label",
            "Mentions predicted label",
        ),
    ]:

        count = int(
            subset_df[
                column
            ].sum()
        )


        audit_rows.append({
            "criterion": description,

            "count": count,

            "percentage": round(
                100.0
                *
                count
                /
                len(
                    subset_df
                ),
                2,
            ),
        })


    print(
        "\nRATIONALE CONTENT AUDIT"
    )

    display(
        pd.DataFrame(
            audit_rows
        )
    )


    # ========================================================
    # REPRESENTATIVE TRACES
    # ========================================================

    print_reasoning_traces(
        subset_df,
        n=n_traces,
    )


# ============================================================
# FINAL AUDIT
# ============================================================

print(
    "Prepared analysis dataframe:",
    analysis_df.shape,
)


print(
    "Original database lookup:",
    len(
        case_lookup
    ),
    "cases",
)


print(
    "Predictions:"
)


display(
    analysis_df[
        "prediction"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "prediction"
    )
    .reset_index(
        name="count"
    )
)


# Quick verification using one case.
sample_case_id = str(
    analysis_df.iloc[
        0
    ][
        "case_id"
    ]
)


sample_temporal_features = get_exact_temporal_features(
    sample_case_id
)


print(
    "\nTemporal-feature lookup verified for:",
    sample_case_id,
)


print(
    "Local feature keys:",
    list(
        sample_temporal_features[
            "local_temporal_features"
        ].keys()
    ),
)


print(
    "Global feature keys:",
    list(
        sample_temporal_features[
            "global_shift_features"
        ].keys()
    ),
)

Prepared analysis dataframe: (400, 29)
Original database lookup: 400 cases
Predictions:


/tmp/ipykernel_704/2386020071.py:302: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ] = lower_reasoning.str.contains(


,prediction,count
0,NORMAL,201
1,ANOMALOUS,199



Temporal-feature lookup verified for: consolidation_normal_000
Local feature keys: ['clean_overlap_seconds', 'clean_overlap_percent', 'signed_strict_offsets_seconds', 'num_signed_strict_offsets', 'offset_mean_seconds', 'offset_median_seconds', 'offset_max_seconds', 'offset_p75_seconds', 'offset_p90_seconds', 'num_offsets_above_1_5_seconds', 'percent_offsets_above_1_5_seconds']
Global feature keys: ['best_B_correction_shift_seconds', 'estimated_B_lateness_seconds', 'alignment_score_gain_vs_zero', 'best_num_bilateral_events', 'best_event_coverage_percent']


In [ ]:
# ============================================================
# 1. CORRECTLY DETECTED LAG CASES
# GOLD ANOMALOUS — PREDICTED ANOMALOUS
# ============================================================

lag_correct_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "lag"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


analyse_subset(
    lag_correct_df,
    title=(
        "CORRECTLY DETECTED LAG CASES "
        "(Gold ANOMALOUS, Predicted ANOMALOUS)"
    ),
    n_traces=10,
)


##############################################################################################################
CORRECTLY DETECTED LAG CASES (Gold ANOMALOUS, Predicted ANOMALOUS)
##############################################################################################################
Number of cases: 43

CASE VARIANT DISTRIBUTION


,case_variant,count
0,lag_3sec,25
1,lag_2sec,18



RATIONALE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,43.00,43.0
mean,198.91,7.0
std,35.22,0.0
min,126.00,7.0
25%,179.50,7.0
50%,201.00,7.0
75%,218.00,7.0
max,272.00,7.0



RATIONALE CONTENT AUDIT


,criterion,count,percentage
0,Mentions participation,43,100.00
1,Mentions local evidence,43,100.00
2,Mentions global evidence,43,100.00
3,Mentions temporal evidence,43,100.00
4,Mentions semantic evidence,43,100.00
5,Mentions weak/limited evidence,15,34.88
6,Mentions predicted label,37,86.05



REPRESENTATIVE MODEL-GENERATED RATIONALES: 10

TRACE 1
CASE ID: consolidation_lag_2sec_033
CASE FAMILY: lag
CASE VARIANT: lag_2sec
GOLD LABEL: ANOMALOUS
PREDICTION: ANOMALOUS

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 5.29,
  "clean_overlap_percent": 4.41,
  "signed_strict_offsets_seconds": [],
  "num_signed_strict_offsets": 0,
  "offset_mean_seconds": null,
  "offset_median_seconds": null,
  "offset_max_seconds": null,
  "offset_p75_seconds": null,
  "offset_p90_seconds": null,
  "num_offsets_above_1_5_seconds": 0,
  "percent_offsets_above_1_5_seconds": null
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": 2.0,
  "estimated_B_lateness_seconds": 0.0,
  "alignment_score_gain_vs_zero": 0.260435,
  "best_num_bilateral_events": 7,
  "best_event_coverage_percent": 38.8889
}

RATIONALE LENGTH
Words: 131
Detected numbered steps: 7

MODEL-GENERATED RATIONALE
1. Participation validity: Both participants speak, which is compatibl

In [ ]:
# ============================================================
# 2. MISSED LAG CASES
# GOLD ANOMALOUS — PREDICTED NORMAL
# ============================================================

lag_missed_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "lag"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


analyse_subset(
    lag_missed_df,
    title=(
        "MISSED LAG CASES "
        "(Gold ANOMALOUS, Predicted NORMAL)"
    ),
    n_traces=10,
)


##############################################################################################################
MISSED LAG CASES (Gold ANOMALOUS, Predicted NORMAL)
##############################################################################################################
Number of cases: 57

CASE VARIANT DISTRIBUTION


,case_variant,count
0,lag_2sec,32
1,lag_3sec,25



RATIONALE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,57.00,57.00
mean,173.12,7.02
std,48.64,0.13
min,128.00,7.00
25%,136.00,7.00
50%,145.00,7.00
75%,225.00,7.00
max,281.00,8.00



RATIONALE CONTENT AUDIT


,criterion,count,percentage
0,Mentions participation,57,100.00
1,Mentions local evidence,57,100.00
2,Mentions global evidence,57,100.00
3,Mentions temporal evidence,57,100.00
4,Mentions semantic evidence,57,100.00
5,Mentions weak/limited evidence,39,68.42
6,Mentions predicted label,57,100.00



REPRESENTATIVE MODEL-GENERATED RATIONALES: 10

TRACE 1
CASE ID: consolidation_lag_2sec_065
CASE FAMILY: lag
CASE VARIANT: lag_2sec
GOLD LABEL: ANOMALOUS
PREDICTION: NORMAL

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 7.15,
  "clean_overlap_percent": 5.96,
  "signed_strict_offsets_seconds": [
    0.57,
    -0.49,
    0.44
  ],
  "num_signed_strict_offsets": 3,
  "offset_mean_seconds": 0.17,
  "offset_median_seconds": 0.44,
  "offset_max_seconds": 0.57,
  "offset_p75_seconds": 0.51,
  "offset_p90_seconds": 0.54,
  "num_offsets_above_1_5_seconds": 0,
  "percent_offsets_above_1_5_seconds": 0.0
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": -0.4,
  "estimated_B_lateness_seconds": 0.4,
  "alignment_score_gain_vs_zero": 0.060752,
  "best_num_bilateral_events": 5,
  "best_event_coverage_percent": 50.0
}

RATIONALE LENGTH
Words: 128
Detected numbered steps: 7

MODEL-GENERATED RATIONALE
1. Participation validity: Both participant

In [ ]:
# ============================================================
# 3. CORRECTLY DETECTED WRONG-PARTNER CASES
# GOLD ANOMALOUS — PREDICTED ANOMALOUS
# ============================================================

wrong_partner_correct_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "wrong_partner"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


analyse_subset(
    wrong_partner_correct_df,
    title=(
        "CORRECTLY DETECTED WRONG-PARTNER CASES "
        "(Gold ANOMALOUS, Predicted ANOMALOUS)"
    ),
    n_traces=10,
)


##############################################################################################################
CORRECTLY DETECTED WRONG-PARTNER CASES (Gold ANOMALOUS, Predicted ANOMALOUS)
##############################################################################################################
Number of cases: 46

CASE VARIANT DISTRIBUTION


,case_variant,count
0,wrong_partner,46



RATIONALE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,46.00,46.0
mean,197.67,7.0
std,34.62,0.0
min,127.00,7.0
25%,177.25,7.0
50%,197.00,7.0
75%,215.25,7.0
max,301.00,7.0



RATIONALE CONTENT AUDIT


,criterion,count,percentage
0,Mentions participation,46,100.00
1,Mentions local evidence,46,100.00
2,Mentions global evidence,46,100.00
3,Mentions temporal evidence,46,100.00
4,Mentions semantic evidence,46,100.00
5,Mentions weak/limited evidence,25,54.35
6,Mentions predicted label,41,89.13



REPRESENTATIVE MODEL-GENERATED RATIONALES: 10

TRACE 1
CASE ID: consolidation_wrong_partner_055
CASE FAMILY: wrong_partner
CASE VARIANT: wrong_partner
GOLD LABEL: ANOMALOUS
PREDICTION: ANOMALOUS

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 18.52,
  "clean_overlap_percent": 15.43,
  "signed_strict_offsets_seconds": [],
  "num_signed_strict_offsets": 0,
  "offset_mean_seconds": null,
  "offset_median_seconds": null,
  "offset_max_seconds": null,
  "offset_p75_seconds": null,
  "offset_p90_seconds": null,
  "num_offsets_above_1_5_seconds": 0,
  "percent_offsets_above_1_5_seconds": null
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": 5.3,
  "estimated_B_lateness_seconds": 0.0,
  "alignment_score_gain_vs_zero": 0.048228,
  "best_num_bilateral_events": 3,
  "best_event_coverage_percent": 16.6667
}

RATIONALE LENGTH
Words: 127
Detected numbered steps: 7

MODEL-GENERATED RATIONALE
1. Participation validity: Both participants spe

In [ ]:
# ============================================================
# 4. MISSED WRONG-PARTNER CASES
# GOLD ANOMALOUS — PREDICTED NORMAL
# ============================================================

wrong_partner_missed_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "wrong_partner"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


analyse_subset(
    wrong_partner_missed_df,
    title=(
        "MISSED WRONG-PARTNER CASES "
        "(Gold ANOMALOUS, Predicted NORMAL)"
    ),
    n_traces=10,
)


##############################################################################################################
MISSED WRONG-PARTNER CASES (Gold ANOMALOUS, Predicted NORMAL)
##############################################################################################################
Number of cases: 54

CASE VARIANT DISTRIBUTION


,case_variant,count
0,wrong_partner,54



RATIONALE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,54.00,54.00
mean,148.41,7.02
std,32.31,0.31
min,112.00,6.00
25%,134.00,7.00
50%,141.00,7.00
75%,146.00,7.00
max,268.00,9.00



RATIONALE CONTENT AUDIT


,criterion,count,percentage
0,Mentions participation,54,100.00
1,Mentions local evidence,54,100.00
2,Mentions global evidence,54,100.00
3,Mentions temporal evidence,54,100.00
4,Mentions semantic evidence,54,100.00
5,Mentions weak/limited evidence,49,90.74
6,Mentions predicted label,54,100.00



REPRESENTATIVE MODEL-GENERATED RATIONALES: 10

TRACE 1
CASE ID: consolidation_wrong_partner_005
CASE FAMILY: wrong_partner
CASE VARIANT: wrong_partner
GOLD LABEL: ANOMALOUS
PREDICTION: NORMAL

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 30.8,
  "clean_overlap_percent": 25.67,
  "signed_strict_offsets_seconds": [
    -0.35,
    0.8
  ],
  "num_signed_strict_offsets": 2,
  "offset_mean_seconds": 0.23,
  "offset_median_seconds": 0.23,
  "offset_max_seconds": 0.8,
  "offset_p75_seconds": 0.51,
  "offset_p90_seconds": 0.69,
  "num_offsets_above_1_5_seconds": 0,
  "percent_offsets_above_1_5_seconds": 0.0
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": 3.1,
  "estimated_B_lateness_seconds": 0.0,
  "alignment_score_gain_vs_zero": 0.210605,
  "best_num_bilateral_events": 7,
  "best_event_coverage_percent": 43.75
}

RATIONALE LENGTH
Words: 112
Detected numbered steps: 6

MODEL-GENERATED RATIONALE
1. Participation validity: Both pa

In [ ]:
# ============================================================
# 5. CORRECTLY DETECTED NORMAL CASES
# GOLD NORMAL — PREDICTED NORMAL
# ============================================================

normal_correct_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "normal"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


analyse_subset(
    normal_correct_df,
    title=(
        "CORRECTLY DETECTED NORMAL CASES "
        "(Gold NORMAL, Predicted NORMAL)"
    ),
    n_traces=10,
)


##############################################################################################################
CORRECTLY DETECTED NORMAL CASES (Gold NORMAL, Predicted NORMAL)
##############################################################################################################
Number of cases: 90

CASE VARIANT DISTRIBUTION


,case_variant,count
0,normal,90



RATIONALE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,90.00,90.00
mean,142.73,6.98
std,24.57,0.15
min,111.00,6.00
25%,133.00,7.00
50%,137.00,7.00
75%,143.00,7.00
max,256.00,7.00



RATIONALE CONTENT AUDIT


,criterion,count,percentage
0,Mentions participation,90,100.00
1,Mentions local evidence,90,100.00
2,Mentions global evidence,90,100.00
3,Mentions temporal evidence,90,100.00
4,Mentions semantic evidence,90,100.00
5,Mentions weak/limited evidence,76,84.44
6,Mentions predicted label,90,100.00



REPRESENTATIVE MODEL-GENERATED RATIONALES: 10

TRACE 1
CASE ID: consolidation_normal_011
CASE FAMILY: normal
CASE VARIANT: normal
GOLD LABEL: NORMAL
PREDICTION: NORMAL

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 9.38,
  "clean_overlap_percent": 7.82,
  "signed_strict_offsets_seconds": [
    -0.51,
    -0.22,
    -0.76,
    0.17,
    0.68,
    1.15,
    -1.85,
    -1.38
  ],
  "num_signed_strict_offsets": 8,
  "offset_mean_seconds": -0.34,
  "offset_median_seconds": -0.36,
  "offset_max_seconds": 1.15,
  "offset_p75_seconds": 0.3,
  "offset_p90_seconds": 0.82,
  "num_offsets_above_1_5_seconds": 0,
  "percent_offsets_above_1_5_seconds": 0.0
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": 0.2,
  "estimated_B_lateness_seconds": 0.0,
  "alignment_score_gain_vs_zero": 0.009634,
  "best_num_bilateral_events": 17,
  "best_event_coverage_percent": 77.2727
}

RATIONALE LENGTH
Words: 111
Detected numbered steps: 6

MODEL-GENERATED

In [ ]:
# ============================================================
# 6. FALSE-POSITIVE NORMAL CASES
# GOLD NORMAL — PREDICTED ANOMALOUS
# ============================================================

normal_false_positive_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "normal"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


analyse_subset(
    normal_false_positive_df,
    title=(
        "FALSE-POSITIVE NORMAL CASES "
        "(Gold NORMAL, Predicted ANOMALOUS)"
    ),
    n_traces=10,
)


##############################################################################################################
FALSE-POSITIVE NORMAL CASES (Gold NORMAL, Predicted ANOMALOUS)
##############################################################################################################
Number of cases: 10

CASE VARIANT DISTRIBUTION


,case_variant,count
0,normal,10



RATIONALE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,10.00,10.0
mean,220.40,7.0
std,47.06,0.0
min,153.00,7.0
25%,182.00,7.0
50%,225.50,7.0
75%,238.25,7.0
max,312.00,7.0



RATIONALE CONTENT AUDIT


,criterion,count,percentage
0,Mentions participation,10,100.0
1,Mentions local evidence,10,100.0
2,Mentions global evidence,10,100.0
3,Mentions temporal evidence,10,100.0
4,Mentions semantic evidence,10,100.0
5,Mentions weak/limited evidence,2,20.0
6,Mentions predicted label,8,80.0



REPRESENTATIVE MODEL-GENERATED RATIONALES: 10

TRACE 1
CASE ID: consolidation_normal_065
CASE FAMILY: normal
CASE VARIANT: normal
GOLD LABEL: NORMAL
PREDICTION: ANOMALOUS

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 7.17,
  "clean_overlap_percent": 5.98,
  "signed_strict_offsets_seconds": [
    0.71
  ],
  "num_signed_strict_offsets": 1,
  "offset_mean_seconds": 0.71,
  "offset_median_seconds": 0.71,
  "offset_max_seconds": 0.71,
  "offset_p75_seconds": 0.71,
  "offset_p90_seconds": 0.71,
  "num_offsets_above_1_5_seconds": 0,
  "percent_offsets_above_1_5_seconds": 0.0
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": 1.4,
  "estimated_B_lateness_seconds": 0.0,
  "alignment_score_gain_vs_zero": 0.193555,
  "best_num_bilateral_events": 5,
  "best_event_coverage_percent": 50.0
}

RATIONALE LENGTH
Words: 153
Detected numbered steps: 7

MODEL-GENERATED RATIONALE
1. Participation validity: Both participants speak, which is consi

In [ ]:
# ============================================================
# 7. CORRECTLY DETECTED SILENT-PARTNER CASES
# GOLD ANOMALOUS — PREDICTED ANOMALOUS
# ============================================================

silent_correct_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "silent_partner"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


analyse_subset(
    silent_correct_df,
    title=(
        "CORRECTLY DETECTED SILENT-PARTNER CASES "
        "(Gold ANOMALOUS, Predicted ANOMALOUS)"
    ),
    n_traces=5,
)


silent_missed_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "silent_partner"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


print(
    "\nMissed silent-partner cases:",
    len(
        silent_missed_df
    ),
)


##############################################################################################################
CORRECTLY DETECTED SILENT-PARTNER CASES (Gold ANOMALOUS, Predicted ANOMALOUS)
##############################################################################################################
Number of cases: 100

CASE VARIANT DISTRIBUTION


,case_variant,count
0,silent_partner,100



RATIONALE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,100.00,100.00
mean,168.67,6.89
std,11.34,0.31
min,138.00,6.00
25%,161.00,7.00
50%,167.00,7.00
75%,174.00,7.00
max,201.00,7.00



RATIONALE CONTENT AUDIT


,criterion,count,percentage
0,Mentions participation,100,100.0
1,Mentions local evidence,100,100.0
2,Mentions global evidence,100,100.0
3,Mentions temporal evidence,100,100.0
4,Mentions semantic evidence,100,100.0
5,Mentions weak/limited evidence,3,3.0
6,Mentions predicted label,100,100.0



REPRESENTATIVE MODEL-GENERATED RATIONALES: 5

TRACE 1
CASE ID: consolidation_silent_partner_000
CASE FAMILY: silent_partner
CASE VARIANT: silent_partner
GOLD LABEL: ANOMALOUS
PREDICTION: ANOMALOUS

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 0.0,
  "clean_overlap_percent": 0.0,
  "signed_strict_offsets_seconds": [],
  "num_signed_strict_offsets": 0,
  "offset_mean_seconds": null,
  "offset_median_seconds": null,
  "offset_max_seconds": null,
  "offset_p75_seconds": null,
  "offset_p90_seconds": null,
  "num_offsets_above_1_5_seconds": 0,
  "percent_offsets_above_1_5_seconds": null
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": -0.0,
  "estimated_B_lateness_seconds": 0.0,
  "alignment_score_gain_vs_zero": 0.0,
  "best_num_bilateral_events": 0,
  "best_event_coverage_percent": 0.0
}

RATIONALE LENGTH
Words: 138
Detected numbered steps: 6

MODEL-GENERATED RATIONALE
1. Participation validity: Participant A speaks, while Par

In [ ]:
# ============================================================
# 8. SUMMARY ACROSS ALL SUBSETS
# ============================================================

analysis_groups = {
    "lag_correct": lag_correct_df,
    "lag_missed": lag_missed_df,
    "wrong_partner_correct": wrong_partner_correct_df,
    "wrong_partner_missed": wrong_partner_missed_df,
    "normal_correct": normal_correct_df,
    "normal_false_positive": normal_false_positive_df,
    "silent_partner_correct": silent_correct_df,
}


summary_rows = []


for group_name, group_df in analysis_groups.items():

    num_cases = len(
        group_df
    )

    summary_rows.append({
        "analysis_group": group_name,
        "num_cases": num_cases,
        "mean_reasoning_words": round(
            group_df[
                "reasoning_word_count"
            ].mean(),
            2,
        ),
        "mean_reasoning_steps": round(
            group_df[
                "reasoning_step_count"
            ].mean(),
            2,
        ),
        "mentions_participation": int(
            group_df[
                "mentions_participation"
            ].sum()
        ),
        "mentions_local": int(
            group_df[
                "mentions_local"
            ].sum()
        ),
        "mentions_global": int(
            group_df[
                "mentions_global"
            ].sum()
        ),
        "mentions_temporal": int(
            group_df[
                "mentions_temporal"
            ].sum()
        ),
        "mentions_semantic": int(
            group_df[
                "mentions_semantic"
            ].sum()
        ),
        "mentions_weak_or_limited": int(
            group_df[
                "mentions_limited_or_weak"
            ].sum()
        ),
        "mentions_predicted_label": int(
            group_df[
                "mentions_predicted_label"
            ].sum()
        ),
    })


subset_summary_df = pd.DataFrame(
    summary_rows
)


display(
    subset_summary_df
)


,analysis_group,num_cases,mean_reasoning_words,mean_reasoning_steps,mentions_participation,mentions_local,mentions_global,mentions_temporal,mentions_semantic,mentions_weak_or_limited,mentions_predicted_label
0,lag_correct,43,198.91,9.23,43,43,43,43,43,15,37
1,lag_missed,57,173.12,8.11,57,57,57,57,57,39,57
2,wrong_partner_correct,46,197.67,9.33,46,46,46,46,46,25,41
3,wrong_partner_missed,54,148.41,7.33,54,54,54,54,54,49,54
4,normal_correct,90,142.73,7.09,90,90,90,90,90,76,90
5,normal_false_positive,10,220.40,9.80,10,10,10,10,10,2,8
6,silent_partner_correct,100,168.67,6.90,100,100,100,100,100,3,100


In [ ]:
from google.colab import runtime

runtime.unassign()